# ⚕️ Sistemas Especialistas Baseados em Regras
### Inteligência Artificial — Ciência da Computação · Aula 08

<a href="https://colab.research.google.com/github/profluiscaparroz/inteligencia-artificial/blob/main/08-sistemas-especialistas/08_Sistemas_Especialistas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

**Objetivo desta aula:** Compreender os fundamentos dos Sistemas Especialistas, sua arquitetura, mecanismos de representação e propagação de incerteza, implementar sistemas completos com fatores de certeza e analisar aplicações clássicas e modernas.

---

### 🎯 Objetivos de aprendizagem

Ao final desta aula, você será capaz de:

1. **Diferenciar** um sistema especialista de um sistema de produção genérico;
2. **Descrever** a arquitetura clássica (base de conhecimento, motor de inferência, memória de trabalho, módulo de explicação, interface);
3. **Construir** um sistema que **pergunta** ao usuário e **explica** seu raciocínio (POR QUÊ? / COMO?);
4. **Calcular** e **interpretar** fatores de certeza (CF): conjunção, propagação e combinação de evidências;
5. **Reconhecer** as limitações dos CFs (dupla contagem, dependência entre evidências) e de bases de regras grandes;
6. **Verificar** automaticamente uma base de conhecimento (redundâncias, conflitos, ciclos).

### 🗺️ Roteiro da aula

| Parte | Conteúdo | Destaques práticos |
|:-:|---|---|
| 0 | Preparação do ambiente | — |
| 1 | O que são sistemas especialistas e história | — |
| 2 | Arquitetura | Mini sistema especialista com perguntas e explicações POR QUÊ/COMO |
| 3 | Representação do conhecimento e fatores de certeza | Gráficos e calculadora interativa de CF 🎛️ |
| 4 | Sistema médico completo com CF | 3 casos clínicos, animação da certeza crescendo 🎬, rede de conhecimento, consultório virtual 🎛️, limitações dos CFs |
| 5 | Estudo de caso: MYCIN | — |
| 6 | Sistemas modernos | Verificação automática de bases de regras |
| 7–10 | Comparações, aplicações, exercícios e referências | — |

> 💡 **Como usar:** execute as células **em ordem**. 🎬 = animação (botão ▶), 🎛️ = controle interativo. Depois de **cada resultado** há uma seção **🔎 Como interpretar**.

---

## Parte 0 — Preparação do ambiente

Execute a célula abaixo antes das demais.

In [ ]:
# =====================================================
# IMPORTAÇÕES E FUNÇÕES AUXILIARES — execute primeiro
# =====================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact

# Animações renderizadas como player HTML/JavaScript (funciona no Colab sem ffmpeg)
plt.rcParams['animation.html'] = 'jshtml'
plt.rcParams['animation.embed_limit'] = 60
plt.rcParams['figure.dpi'] = 90

CORES = {'azul': '#1f77b4', 'vermelho': '#d62728', 'verde': '#2ca02c', 'laranja': '#ff7f0e',
         'roxo': '#9467bd', 'cinza': '#bbbbbb', 'amarelo': '#ffd54f'}


def mostrar_animacao(anim):
    # Exibe uma animação do matplotlib como player interativo (▶ ⏸ ⏭) e fecha a figura estática.
    plt.close(anim._fig)
    display(HTML(anim.to_jshtml()))


print('✅ Ambiente pronto!')

#### 🔎 Como interpretar esta célula

- Carregamos `numpy`, `matplotlib` (gráficos e animações) e `ipywidgets` (controles interativos).
- `mostrar_animacao(anim)` exibe as animações como um **player** (▶ ⏸ ⏭). Use o avanço quadro a quadro para estudar cada passo.

## 1. O que são Sistemas Especialistas?

**Sistemas Especialistas** (Expert Systems) são programas de computador que emulam a capacidade de tomada de decisão de um especialista humano em um domínio específico e restrito. Representam um dos ramos mais bem-sucedidos da Inteligência Artificial clássica, especialmente durante os anos 1970-1990.

### 1.1 Definição Formal

Segundo **Feigenbaum** (1977), considerado o "pai dos sistemas especialistas":

> *"Um sistema especialista é um programa de computador inteligente que usa conhecimento e procedimentos de inferência para resolver problemas que são suficientemente difíceis para requererem significativa expertise humana."*

**Jackson** (1999) define características essenciais:

1. **Alto desempenho:** Resolvem problemas em nível comparável ou superior a especialistas humanos
2. **Domínio restrito:** Expertise limitada a área específica (medicina, finanças, geologia)
3. **Conhecimento explícito:** Conhecimento separado do mecanismo de inferência
4. **Raciocínio transparente:** Capacidade de explicar suas conclusões
5. **Tratamento de incerteza:** Lidam com informações incompletas ou imprecisas

### 1.2 Diferença entre Sistemas Especialistas e Sistemas de Produção

Embora relacionados, há distinções importantes:

| **Aspecto** | **Sistemas de Produção** | **Sistemas Especialistas** |
|-------------|-------------------------|---------------------------|
| **Escopo** | Arquitetura geral de resolução de problemas | Aplicação específica com expertise em domínio |
| **Conhecimento** | Regras procedurais genéricas | Conhecimento heurístico especializado |
| **Incerteza** | Geralmente determinístico | Fatores de certeza, probabilidades |
| **Explicação** | Traço de execução simples | Justificativa detalhada do raciocínio |
| **Aquisição** | Programação direta | Engenharia de conhecimento com especialistas |
| **Exemplos** | GPS, SOAR | MYCIN, DENDRAL, XCON |

**Em resumo:** Todo sistema especialista usa um sistema de produção como base, mas adiciona componentes de manipulação de incerteza, explicação sofisticada e conhecimento de domínio profundo extraído de especialistas humanos.

### 1.3 Breve História dos Sistemas Especialistas

A evolução dos sistemas especialistas marca uma era dourada da IA simbólica:

- **1965-1972 — DENDRAL (Stanford):** Primeiro sistema especialista verdadeiro, desenvolvido por **Feigenbaum**, **Buchanan** e **Lederberg**. Identificava estruturas moleculares a partir de dados de espectrometria de massa. Demonstrou que conhecimento heurístico especializado supera algoritmos gerais.

- **1972-1980 — MYCIN (Stanford):** Desenvolvido por **Edward Shortliffe** sob orientação de **Buchanan**. Diagnosticava infecções sanguíneas e recomendava antibióticos. Introduziu **fatores de certeza (CF)** para raciocínio sob incerteza. Em uma avaliação cega (Yu et al., 1979), suas recomendações de terapia foram consideradas aceitáveis em 65% dos casos — resultado igual ou superior ao dos especialistas de Stanford avaliados no mesmo estudo.

- **1976 — PROSPECTOR:** Sistema para exploração mineral desenvolvido por **Duda** e **Gashnig** na SRI International. Usava raciocínio bayesiano. Famoso por prever depósito de molibdênio avaliado em $100 milhões.

- **1980 — XCON/R1 (Digital Equipment Corp.):** Desenvolvido por **John McDermott** na CMU. Configurava sistemas de computadores VAX. Economizou milhões de dólares à DEC. Primeiro sistema especialista comercialmente bem-sucedido em larga escala (~2500 regras, processava 80 pedidos/dia).

- **1981 — CADUCEUS (INTERNIST-I):** Sistema para diagnóstico médico geral desenvolvido por **Myers** e **Pople** na Universidade de Pittsburgh. Base de conhecimento com ~500 doenças e ~3500 sintomas.

- **Anos 1980 — Era de Ouro:** Explosão de sistemas especialistas comerciais. Empresas investiram bilhões em sistemas para diagnóstico, planejamento, configuração. Surgem shells (EMYCIN, KEE, ART) — frameworks para construir sistemas especialistas.

- **1985-1990 — Inverno da IA:** Limitações se tornaram aparentes:
  - **Aquisição de conhecimento:** Extrair conhecimento de especialistas é caro e demorado
  - **Manutenção:** Bases de conhecimento grandes são frágeis e difíceis de atualizar
  - **Escopo limitado:** Performance degrada fora do domínio específico
  - **Incapacidade de aprender:** Não melhoram com experiência

- **Anos 1990-2000 — Declínio e Integração:** Sistemas especialistas puros deram lugar a abordagens híbridas, combinando regras com redes neurais, raciocínio probabilístico e aprendizado de máquina.

- **Atualidade — Renascimento Parcial:** Sistemas especialistas modernos integrados com ML:
  - **Motores de regras de negócio:** Drools, CLIPS continuam em uso industrial
  - **IA Explicável (XAI):** Ressurgimento do interesse em sistemas transparentes
  - **Sistemas híbridos:** Aprendizado profundo para reconhecimento + regras para raciocínio
  - **IBM Watson:** Combina ML, processamento de linguagem natural e raciocínio baseado em conhecimento

## 2. Arquitetura de um Sistema Especialista

A arquitetura clássica de um sistema especialista, formalizada por **Buchanan** e **Shortliffe** (1984), consiste em:

```
┌─────────────────────────────────────────────────────────────┐
│                   SISTEMA ESPECIALISTA                      │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────────────┐      ┌───────────────────────┐   │
│  │  BASE DE CONHECIMENTO│◄────►│  MOTOR DE INFERÊNCIA  │   │
│  │   (Knowledge Base)  │      │  (Inference Engine)   │   │
│  │                     │      │                       │   │
│  │  • Regras           │      │  • Forward Chaining   │   │
│  │  • Fatos            │      │  • Backward Chaining  │   │
│  │  • Frames           │      │  • Propagação de CF   │   │
│  │  • Fatores Certeza  │      │  • Resolução Conflitos│   │
│  └─────────────────────┘      └───────────────────────┘   │
│           ▲                             │                  │
│           │                             ▼                  │
│  ┌────────┴──────────┐       ┌─────────────────────┐     │
│  │  MEMÓRIA TRABALHO │       │ MÓDULO DE EXPLICAÇÃO│     │
│  │ (Working Memory)  │       │ (Explanation Module)│     │
│  │                   │       │                     │     │
│  │ • Fatos conhecidos│       │ • Por quê? (WHY)    │     │
│  │ • Hipóteses atuais│       │ • Como? (HOW)       │     │
│  │ • Valores CF      │       │ • Justificativas    │     │
│  └───────────────────┘       └─────────────────────┘     │
│           ▲                             │                  │
└───────────┼─────────────────────────────┼──────────────────┘
            │                             ▼
   ┌────────┴────────────┐      ┌──────────────────┐
   │ INTERFACE DE USUÁRIO│◄────►│   ESPECIALISTA   │
   │  (User Interface)   │      │   (Acquisition)  │
   │                     │      │                  │
   │ • Perguntas/Respos. │      │ • Eng. Conhec.   │
   │ • Explicações       │      │ • Atualização KB │
   └─────────────────────┘      └──────────────────┘
```

### 2.1 Componentes Principais

#### 2.1.1 Base de Conhecimento (Knowledge Base)

Armazena conhecimento especializado do domínio. Composta por:

- **Regras de produção:**
  ```
  SE <premissas> ENTÃO <conclusão> [CF]
  ```
  
- **Fatos:** Informações definitivas sobre o domínio
  
- **Frames:** Estruturas para representar objetos e hierarquias
  
- **Metarregras:** Regras sobre como usar outras regras

#### 2.1.2 Motor de Inferência (Inference Engine)

Mecanismo de raciocínio que manipula conhecimento. Implementa:

- **Forward Chaining (Encadeamento Para Frente):** Dos dados às conclusões
- **Backward Chaining (Encadeamento Para Trás):** Das hipóteses aos dados
- **Propagação de incerteza:** Calcula fatores de certeza combinados
- **Resolução de conflitos:** Decide qual regra disparar

#### 2.1.3 Memória de Trabalho (Working Memory)

Armazena estado atual da consulta:
- Fatos observados
- Hipóteses intermediárias
- Valores de certeza

#### 2.1.4 Módulo de Explicação

Justifica raciocínio do sistema. Responde:

- **"POR QUÊ?"** — Por que o sistema fez determinada pergunta?
- **"COMO?"** — Como o sistema chegou a determinada conclusão?

#### 2.1.5 Interface de Usuário

Facilita interação entre sistema e humano (usuário final ou especialista).

#### 2.1.6 Subsistema de Aquisição de Conhecimento

Permite que especialistas adicionem/modifiquem regras sem programação.

### 2.2 Exemplo mínimo: um "técnico de informática" que pergunta e se explica

Antes do sistema médico completo, vamos ver os componentes da arquitetura funcionando em um exemplo pequeno. Ele faz o que o MYCIN fazia:

- **Raciocina para trás** (*backward chaining*): parte das **hipóteses** (fonte com defeito? memória RAM? placa de vídeo?);
- **Pergunta** ao usuário só o que precisa saber (aqui as respostas são simuladas por um dicionário);
- Responde **POR QUÊ** está fazendo cada pergunta (a regra que está tentando aplicar);
- Responde **COMO** chegou à conclusão (a cadeia de regras e respostas).

```
R1: SE computador_nao_liga E led_da_fonte_apagado              ENTÃO falta_energia
R2: SE falta_energia E tomada_funciona E cabo_bem_conectado    ENTÃO fonte_com_defeito
R3: SE computador_nao_liga E ventoinha_gira E bipes_ao_ligar   ENTÃO problema_memoria_ram
R4: SE liga_mas_tela_preta E monitor_funciona_em_outro_pc      ENTÃO problema_placa_de_video
```

In [ ]:
# Um sistema especialista MÍNIMO, no estilo do MYCIN:
#   • backward chaining (parte das hipóteses);
#   • PERGUNTA ao usuário apenas os fatos necessários;
#   • explica POR QUÊ pergunta e COMO chegou à conclusão.
REGRAS_PC = [
    ('R1', ['computador_nao_liga', 'led_da_fonte_apagado'], 'falta_energia'),
    ('R2', ['falta_energia', 'tomada_funciona', 'cabo_bem_conectado'], 'fonte_com_defeito'),
    ('R3', ['computador_nao_liga', 'ventoinha_gira', 'bipes_ao_ligar'], 'problema_memoria_ram'),
    ('R4', ['liga_mas_tela_preta', 'monitor_funciona_em_outro_pc'], 'problema_placa_de_video'),
]
HIPOTESES = ['fonte_com_defeito', 'problema_memoria_ram', 'problema_placa_de_video']


class MiniEspecialista:
    def __init__(self, regras, respostas_do_usuario):
        self.regras = regras
        self.respostas = respostas_do_usuario     # simula o usuário respondendo sim/não
        self.fatos, self.como, self.pilha = {}, {}, []
        self.n_perguntas = 0

    def perguntar(self, fato):
        self.n_perguntas += 1
        print(f'  🤖 {fato.replace("_", " ")}? ', end='')
        if self.pilha:
            nome, conds, concl = self.pilha[-1]
            print(f'\n     (POR QUÊ? Estou tentando a regra {nome} para concluir "{concl}", '
                  f'que precisa de: {", ".join(conds)})', end='')
        resposta = self.respostas.get(fato, False)
        print(f'\n  👤 {"sim" if resposta else "não"}')
        return resposta

    def provar(self, meta):
        if meta in self.fatos:
            return self.fatos[meta]
        regras_meta = [r for r in self.regras if r[2] == meta]
        if not regras_meta:                       # fato observável: perguntar
            self.fatos[meta] = self.perguntar(meta)
            return self.fatos[meta]
        for regra in regras_meta:
            self.pilha.append(regra)
            ok = all(self.provar(c) for c in regra[1])
            self.pilha.pop()
            if ok:
                self.fatos[meta], self.como[meta] = True, regra
                return True
        self.fatos[meta] = False
        return False

    def explicar_como(self, fato, nivel=0):
        recuo = '    ' * nivel
        if fato in self.como:
            nome, conds, _ = self.como[fato]
            print(f'{recuo}• {fato}  ← regra {nome}, porque:')
            for c in conds:
                self.explicar_como(c, nivel + 1)
        else:
            print(f'{recuo}• {fato}  ← informado pelo usuário')

    def diagnosticar(self, hipoteses):
        for h in hipoteses:
            print(f'\n🎯 Investigando a hipótese: {h}')
            if self.provar(h):
                print(f'\n✅ DIAGNÓSTICO: {h.replace("_", " ").upper()}')
                print('\n💬 COMO cheguei a essa conclusão?')
                self.explicar_como(h)
                return h
        print('\n❓ Nenhuma hipótese confirmada.')


print('=' * 70, '\nCASO A — o computador está totalmente "morto"\n' + '=' * 70)
caso_a = MiniEspecialista(REGRAS_PC, {'computador_nao_liga': True, 'led_da_fonte_apagado': True,
                                      'tomada_funciona': True, 'cabo_bem_conectado': True})
caso_a.diagnosticar(HIPOTESES)

#### 🔎 Como interpretar o resultado

O diálogo reproduz, em miniatura, uma consulta ao **MYCIN**:

1. **🎯 Investigando a hipótese:** o sistema **não** pede todos os sintomas de uma vez. Ele escolhe uma **hipótese** (`fonte_com_defeito`) e raciocina **para trás**: "para concluir isso, preciso de quê?".
2. **🤖 Pergunta + POR QUÊ?:** cada pergunta vem com a justificativa, que é a **regra** sendo tentada no momento. Esse é o **módulo de explicação** respondendo à pergunta *"por que você quer saber isso?"*. Repare que as duas primeiras perguntas se justificam pela regra **R1** (uma submeta) e as duas últimas pela **R2** (a meta principal). O sistema sabe **em que ponto do raciocínio** está, porque mantém uma **pilha** de regras em andamento.
3. **✅ Diagnóstico + COMO?:** a explicação final é uma **árvore**: `fonte_com_defeito` foi concluída pela R2, que usou `falta_energia` (concluída pela R1) e duas respostas do usuário.

- **Por que isso importa?** Um médico (ou técnico) não aceita uma recomendação sem justificativa. A capacidade de explicar **POR QUÊ** e **COMO** foi uma das principais razões da aceitação dos sistemas especialistas e hoje é exigida por regulamentações como a LGPD para decisões automatizadas.
- **💡 Conceito-chave:** a explicação **não foi programada à parte**. Ela é gerada automaticamente a partir das **próprias regras**. É a vantagem de ter o conhecimento **explícito** e **separado** do motor.

In [ ]:
print('=' * 70, '\nCASO B — o computador não liga, mas a ventoinha gira e ele apita\n' + '=' * 70)
caso_b = MiniEspecialista(REGRAS_PC, {'computador_nao_liga': True, 'led_da_fonte_apagado': False,
                                      'ventoinha_gira': True, 'bipes_ao_ligar': True})
caso_b.diagnosticar(HIPOTESES)
print(f'\nPerguntas feitas — caso A: {caso_a.n_perguntas} | caso B: {caso_b.n_perguntas} '
      f'(de {len({p for r in REGRAS_PC for p in r[1]} - {r[2] for r in REGRAS_PC})} fatos observáveis possíveis)')

#### 🔎 Como interpretar o resultado

- **A primeira hipótese falhou:** o usuário respondeu **não** para "led da fonte apagado?". Pelo curto-circuito, R1 falha, `falta_energia` não pode ser concluída, e o sistema **abandona** `fonte_com_defeito` **sem perguntar** sobre tomada e cabo, perguntas que seriam inúteis.
- **Segunda hipótese:** ao investigar `problema_memoria_ram`, o sistema **não pergunta de novo** "computador não liga?". A resposta foi **guardada** na memória de trabalho. Só pergunta o que ainda não sabe: ventoinha e bipes.
- **Contagem:** cada caso exigiu só **4 das 8** perguntas possíveis. Em um sistema real com centenas de sintomas, essa economia é enorme. Era por isso que o MYCIN usava *backward chaining*: em medicina, cada informação pode custar um exame.
- **💡 Experimente:** monte um caso C para a placa de vídeo (`liga_mas_tela_preta` e `monitor_funciona_em_outro_pc` = `True`, e `computador_nao_liga` = `False`). Conte as perguntas e explique por que a hipótese de memória RAM é descartada **sem nenhuma pergunta nova**.

## 3. Representação de Conhecimento

### 3.1 Regras de Produção com Fatores de Certeza

A forma básica de uma regra em sistema especialista é:

```
SE <premissa_1> E <premissa_2> E ... E <premissa_n>
ENTÃO <conclusão> [CF = valor]
```

**Exemplo (MYCIN):**
```
REGRA 052:
SE:
  (1) Local da cultura é sangue, E
  (2) Gram do organismo é negativo, E
  (3) Morfologia do organismo é bastonete, E
  (4) Paciente tem histórico de queimadura grave
ENTÃO:
  Há evidência sugestiva (0.4) de que identidade do
  organismo é Pseudomonas
```

### 3.2 Fatores de Certeza (Certainty Factors)

Desenvolvidos para MYCIN por **Shortliffe** e **Buchanan** (1975) como alternativa à probabilidade bayesiana, que se mostrou computacionalmente cara e difícil para especialistas especificarem.

**Definição:** CF é um número no intervalo **[-1, 1]** que representa:

- **CF = 1.0:** Certeza absoluta de que hipótese é verdadeira
- **CF = 0.5:** Evidência sugestiva a favor
- **CF = 0.0:** Sem informação
- **CF = -0.5:** Evidência sugestiva contra
- **CF = -1.0:** Certeza absoluta de que hipótese é falsa

**Interpretação intuitiva:**

```python
CF > 0.8   →  "Definitivamente"
0.5 < CF ≤ 0.8  →  "Provavelmente"
0.2 < CF ≤ 0.5  →  "Talvez"
-0.2 ≤ CF ≤ 0.2  →  "Desconhecido"
CF < -0.2  →  "Provavelmente não"
```

### 3.3 Fórmulas de Combinação de CF

**Propriedades matemáticas dos CF:**

#### 3.3.1 Combinação de Premissas (Conjunção)

Quando regra tem múltiplas premissas conectadas por **E**:

```
CF_combinado(A ∧ B) = min(CF(A), CF(B))
```

**Justificativa:** Cadeia é tão forte quanto seu elo mais fraco.

#### 3.3.2 Propagação através de Regras

Se regra tem CF da regra = CF_regra e premissas têm CF_premissas:

```
CF_conclusão = CF_regra × CF_premissas
```

**Exemplo:**
```
REGRA: SE febre ENTÃO gripe [CF = 0.6]
Observação: febre [CF = 0.8]
Conclusão: gripe [CF = 0.6 × 0.8 = 0.48]
```

#### 3.3.3 Combinação de Evidências para Mesma Hipótese

Quando múltiplas regras concluem a mesma hipótese com diferentes CFs:

**Caso 1:** Ambas evidências positivas (CF₁ > 0, CF₂ > 0)
```
CF_combinado = CF₁ + CF₂ - (CF₁ × CF₂)
           = CF₁ + CF₂(1 - CF₁)
```

**Caso 2:** Ambas evidências negativas (CF₁ < 0, CF₂ < 0)
```
CF_combinado = CF₁ + CF₂ + (CF₁ × CF₂)
```

**Caso 3:** Evidências conflitantes (sinais opostos)
```
CF_combinado = (CF₁ + CF₂) / (1 - min(|CF₁|, |CF₂|))
```

**Propriedades importantes:**
- Comutativa: CF(A, B) = CF(B, A)
- Associativa: CF(CF(A, B), C) = CF(A, CF(B, C))
- Evidência adicional nunca diminui certeza (se mesmo sinal)
- Nunca ultrapassa [-1, 1]

### 3.4 Frames (Esquemas)

Estrutura para representar objetos e suas propriedades. Proposto por **Minsky** (1975).

**Exemplo em sistema médico:**
```
FRAME: Paciente
  - Nome: [string]
  - Idade: [número]
  - Sintomas: [lista]
  - Histórico: [lista]
  - Diagnóstico: [frame Doença]
  
FRAME: Doença
  - Nome: [string]
  - Sintomas-típicos: [lista]
  - Tratamento: [string]
  - CF-diagnóstico: [float]
```

### 3.5 Lógica Fuzzy (Introdução)

Proposta por **Lotfi Zadeh** (1965), lógica fuzzy permite representar conceitos graduais.

**Diferença de CF:**
- **CF:** Incerteza sobre verdade de proposição ("Quão certo estou que paciente tem gripe?")
- **Fuzzy:** Grau de pertinência a conjunto ("Quão alto é o paciente?")

**Exemplo:**
```
Febre-Baixa(37.5°C) = 0.5
Febre-Baixa(38.0°C) = 1.0
Febre-Baixa(39.0°C) = 0.3

Febre-Alta(37.5°C) = 0.0
Febre-Alta(38.5°C) = 0.5
Febre-Alta(39.5°C) = 1.0
```

Sistemas modernos frequentemente combinam CF e lógica fuzzy.

### 3.6 Visualizando os fatores de certeza

Vamos **ver** as fórmulas da Seção 3.3 em ação antes de usá-las no sistema completo.

In [ ]:
def comb(cf1, cf2):                     # mesma fórmula do MYCIN (implementada na Seção 4)
    if cf1 > 0 and cf2 > 0:
        return cf1 + cf2 - cf1 * cf2
    if cf1 < 0 and cf2 < 0:
        return cf1 + cf2 + cf1 * cf2
    den = 1 - min(abs(cf1), abs(cf2))
    return 0.0 if den == 0 else (cf1 + cf2) / den

fig, axs = plt.subplots(1, 2, figsize=(14, 4.6))

# (a) acumulando evidências independentes de mesma força
for cf, cor in [(0.2, CORES['azul']), (0.5, CORES['laranja']), (0.8, CORES['verde'])]:
    acumulado, serie = 0.0, []
    for n in range(1, 11):
        acumulado = cf if n == 1 else comb(acumulado, cf)
        serie.append(acumulado)
    axs[0].plot(range(1, 11), serie, 'o-', color=cor, label=f'cada evidência com CF = {cf}')
axs[0].axhline(1, color='k', ls=':')
axs[0].set_xlabel('nº de evidências a favor'); axs[0].set_ylabel('CF combinado')
axs[0].set_title('(a) Mais evidências → mais certeza, mas nunca passa de 1'); axs[0].legend(fontsize=9)

# (b) mapa de combinação de duas evidências
g = np.linspace(-0.99, 0.99, 199)
Z = np.array([[comb(a, b) for a in g] for b in g])
im = axs[1].imshow(Z, extent=[-1, 1, -1, 1], origin='lower', cmap='RdBu', vmin=-1, vmax=1)
cs = axs[1].contour(g, g, Z, levels=[-0.8, -0.5, 0, 0.5, 0.8], colors='k', linewidths=0.8)
axs[1].clabel(cs, fontsize=8)
axs[1].set_xlabel('CF₁'); axs[1].set_ylabel('CF₂'); plt.colorbar(im, ax=axs[1], label='CF combinado')
axs[1].set_title('(b) Combinação de duas evidências (azul = a favor, vermelho = contra)')
plt.tight_layout(); plt.show()

#### 🔎 Como interpretar os gráficos

**(a) Acumulando evidências a favor**
- Cada curva mostra o CF combinado depois de 1, 2, ..., 10 evidências independentes de mesma força.
- Com evidências **fortes** (0,8), duas já levam a 0,96. Com evidências **fracas** (0,2), dez evidências chegam a ≈ 0,89.
- **Por que nunca passa de 1?** A fórmula `CF₁ + CF₂(1 − CF₁)` soma apenas uma **fração do que falta** para chegar a 1. É como encher um copo despejando sempre uma porcentagem do espaço vazio: ele se aproxima da borda, mas nunca transborda.

**(b) Mapa da combinação de duas evidências**
- **Canto superior direito (azul forte):** duas evidências a favor reforçam a certeza.
- **Canto inferior esquerdo (vermelho forte):** duas evidências contra reforçam a descrença.
- **Diagonal secundária (branco, linha "0"):** evidências de **mesma força e sinais opostos** se **anulam** (por exemplo, +0,6 e −0,6 dão 0).
- Repare como as curvas de nível ficam "apertadas" perto dos cantos: quando já há muita certeza, uma nova evidência a favor muda pouco o resultado.

In [ ]:
# 🎛️ Combine duas evidências e veja qual fórmula é usada
def combinar_interativo(cf1=0.6, cf2=0.5):
    r = comb(cf1, cf2)
    if cf1 > 0 and cf2 > 0:
        caso, formula = 'ambas A FAVOR', f'{cf1:.2f} + {cf2:.2f} − {cf1:.2f}×{cf2:.2f}'
    elif cf1 < 0 and cf2 < 0:
        caso, formula = 'ambas CONTRA', f'{cf1:.2f} + {cf2:.2f} + {cf1:.2f}×{cf2:.2f}'
    else:
        caso, formula = 'CONFLITANTES', f'({cf1:.2f} + {cf2:.2f}) / (1 − min(|{cf1:.2f}|, |{cf2:.2f}|))'
    fig, ax = plt.subplots(figsize=(8, 2.4))
    nomes, vals = ['evidência 1', 'evidência 2', 'COMBINADO'], [cf1, cf2, r]
    ax.barh(nomes, vals, color=[CORES['verde'] if v > 0 else CORES['vermelho'] for v in vals])
    ax.axvline(0, color='k'); ax.set_xlim(-1, 1)
    ax.set_title(f'Caso: {caso}   →   CF = {formula} = {r:+.3f}', fontsize=10)
    plt.show()

interact(combinar_interativo,
         cf1=widgets.FloatSlider(min=-1, max=1, step=0.05, value=0.6, description='CF₁'),
         cf2=widgets.FloatSlider(min=-1, max=1, step=0.05, value=0.5, description='CF₂'));

#### 🔎 Como usar e interpretar

Teste estes casos e compare com o que a intuição diria:

| CF₁ | CF₂ | Resultado | Leitura |
|:-:|:-:|:-:|---|
| 0,6 | 0,5 | **0,80** | Duas evidências moderadas juntas dão uma evidência forte |
| 0,7 | −0,4 | **0,50** | A evidência contrária "desconta" parte da certeza, mas a favorável prevalece |
| 0,9 | −0,9 | **0,00** | Evidências igualmente fortes e opostas se anulam |
| 1,0 | −0,5 | **1,00** | Certeza **absoluta** não pode ser abalada (o denominador vira 1 − 0,5 e o resultado é 1) |
| 1,0 | −1,0 | **0** (indefinido) | Contradição total: a fórmula divide por zero e a implementação devolve 0 por convenção |

- **💡 Conceito-chave:** CF não é probabilidade. Ele mede a **mudança de crença** causada pela evidência (CF = MB − MD). Por isso existe a escala negativa e por isso dois CFs não somam 1.

## 4. Implementação Completa: Sistema Especialista Médico com Fatores de Certeza

Implementaremos um sistema especialista para diagnóstico de doenças respiratórias, inspirado no MYCIN, incluindo:

- Representação de regras com CF
- Propagação de incerteza
- Forward e backward chaining
- Módulo de explicação

### 4.1 Estruturas de Dados Básicas

In [ ]:
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
import math

@dataclass
class Fato:
    """
    Representa um fato com seu fator de certeza.
    
    Atributos:
    ----------
    nome : str
        Nome do fato (ex: 'febre', 'tosse', 'gripe')
    cf : float
        Fator de certeza no intervalo [-1, 1]
    """
    nome: str
    cf: float = 0.0
    
    def __post_init__(self):
        # Garante que CF está no intervalo válido [-1, 1]
        # Em produção, considere avisar usuário se valor original estava fora do intervalo
        self.cf = max(-1.0, min(1.0, self.cf))
    
    def __str__(self):
        return f"{self.nome} [CF={self.cf:.2f}]"
    
    def __hash__(self):
        return hash(self.nome)
    
    def __eq__(self, other):
        if isinstance(other, Fato):
            return self.nome == other.nome
        return False


@dataclass
class Regra:
    """
    Representa uma regra de produção com fator de certeza.
    
    Formato: SE <premissas> ENTÃO <conclusao> [CF]
    
    Atributos:
    ----------
    nome : str
        Identificador único da regra
    premissas : List[str]
        Lista de nomes de fatos que devem ser verdadeiros
    conclusao : str
        Nome do fato concluído
    cf_regra : float
        Fator de certeza da própria regra
    explicacao : str
        Justificativa em linguagem natural
    """
    nome: str
    premissas: List[str]
    conclusao: str
    cf_regra: float
    explicacao: str = ""
    
    def __str__(self):
        premissas_str = " E ".join(self.premissas)
        return f"{self.nome}: SE {premissas_str} ENTÃO {self.conclusao} [CF={self.cf_regra:.2f}]"


print("✓ Estruturas básicas definidas")
print("\nExemplo de uso:")
fato = Fato("febre_alta", 0.8)
print(f"  {fato}")

regra = Regra(
    nome="R1",
    premissas=["febre_alta", "tosse_seca"],
    conclusao="gripe",
    cf_regra=0.7,
    explicacao="Febre alta com tosse seca é sintoma característico de gripe"
)
print(f"  {regra}")

#### 🔎 Como interpretar o resultado

- **`Fato`** guarda um nome e um CF. Repare que `__post_init__` **limita** o CF ao intervalo [−1, 1]: se alguém informar 1,5, o valor vira 1,0.
- **`Regra`** guarda premissas, conclusão, o **CF da regra** (quanto o especialista confia nela) e uma **explicação em linguagem natural**, que será usada para justificar diagnósticos.
- Na saída, a regra de exemplo é lida assim: *"SE febre alta E tosse seca, ENTÃO gripe, com força 0,7"*. O CF 0,7 **não** é a probabilidade de gripe; é a força da associação na opinião do especialista.

In [ ]:
def combinar_cf_conjuncao(cf_list: List[float]) -> float:
    """
    Combina CFs de premissas conectadas por E (conjunção).
    
    Fórmula: CF(A ∧ B ∧ C) = min(CF(A), CF(B), CF(C))
    
    Justificativa: Cadeia é tão forte quanto elo mais fraco.
    """
    if not cf_list:
        return 0.0
    return min(cf_list)


def propagar_cf_regra(cf_regra: float, cf_premissas: float) -> float:
    """
    Propaga CF através de uma regra.
    
    Fórmula: CF(conclusão) = CF(regra) × CF(premissas)
    
    Parâmetros:
    -----------
    cf_regra : float
        Fator de certeza da própria regra
    cf_premissas : float
        CF combinado das premissas
    
    Retorna:
    --------
    float : CF da conclusão
    """
    return cf_regra * cf_premissas


def combinar_cf_multiplas_evidencias(cf1: float, cf2: float) -> float:
    """
    Combina dois CFs que suportam a mesma hipótese.
    
    Implementa fórmulas de MYCIN para combinar evidências:
    
    Caso 1 — Ambos positivos:
        CF = CF₁ + CF₂ - (CF₁ × CF₂)
    
    Caso 2 — Ambos negativos:
        CF = CF₁ + CF₂ + (CF₁ × CF₂)
    
    Caso 3 — Sinais opostos:
        CF = (CF₁ + CF₂) / (1 - min(|CF₁|, |CF₂|))
    
    Referência: Shortliffe & Buchanan (1975)
    """
    # Caso: ambos positivos
    if cf1 > 0 and cf2 > 0:
        return cf1 + cf2 - (cf1 * cf2)
    
    # Caso: ambos negativos
    elif cf1 < 0 and cf2 < 0:
        return cf1 + cf2 + (cf1 * cf2)
    
    # Caso: sinais opostos
    else:
        denominador = 1 - min(abs(cf1), abs(cf2))
        if denominador == 0:
            return 0.0
        return (cf1 + cf2) / denominador


def combinar_cf_lista(cf_list: List[float]) -> float:
    """
    Combina uma lista de CFs aplicando combinação sucessiva.
    
    Propriedade: Operação é comutativa e associativa.
    """
    if not cf_list:
        return 0.0
    
    resultado = cf_list[0]
    for cf in cf_list[1:]:
        resultado = combinar_cf_multiplas_evidencias(resultado, cf)
    
    return resultado


# Testes das funções
print("="*60)
print("TESTES DAS FUNÇÕES DE COMBINAÇÃO DE CF")
print("="*60)

print("\n1. Conjunção (AND):")
print(f"   min([0.8, 0.6, 0.9]) = {combinar_cf_conjuncao([0.8, 0.6, 0.9]):.2f}")
print("   → Elo mais fraco determina força da cadeia")

print("\n2. Propagação através de regra:")
print(f"   Regra CF=0.7, Premissas CF=0.8")
print(f"   Conclusão CF = {propagar_cf_regra(0.7, 0.8):.2f}")

print("\n3. Múltiplas evidências positivas:")
cf_comb = combinar_cf_multiplas_evidencias(0.6, 0.5)
print(f"   CF₁=0.6, CF₂=0.5 → CF={cf_comb:.2f}")
print(f"   Note: {cf_comb:.2f} > max(0.6, 0.5) — evidência reforçada!")

print("\n4. Evidências conflitantes:")
cf_conf = combinar_cf_multiplas_evidencias(0.7, -0.4)
print(f"   CF₁=0.7 (a favor), CF₂=-0.4 (contra) → CF={cf_conf:.2f}")
print(f"   Evidência positiva domina ligeiramente")

print("\n5. Múltiplas evidências encadeadas:")
cf_final = combinar_cf_lista([0.5, 0.6, 0.4])
print(f"   Combinar [0.5, 0.6, 0.4] = {cf_final:.2f}")
print("   → Certeza aumenta gradualmente com mais evidências")

#### 🔎 Como interpretar o resultado

| Teste | Conta | Por que faz sentido |
|---|---|---|
| 1. Conjunção | min(0,8; 0,6; 0,9) = **0,6** | Uma corrente é tão forte quanto seu **elo mais fraco**. Se não temos certeza de um dos sintomas exigidos, não podemos ter mais certeza da combinação |
| 2. Propagação | 0,7 × 0,8 = **0,56** | A dúvida sobre as premissas (0,8) **reduz** a força da regra (0,7). Uma regra "forte" aplicada a dados incertos dá conclusão incerta |
| 3. Duas a favor | 0,6 + 0,5 − 0,30 = **0,80** | Duas evidências independentes **se reforçam**: o resultado supera a maior delas |
| 4. Conflitantes | (0,7 − 0,4)/(1 − 0,4) = **0,50** | A evidência contrária reduz a certeza, mas não a anula |
| 5. Três a favor | 0,5 → 0,8 → **0,88** | Cada nova evidência acrescenta menos (retornos decrescentes) |

- **💡 Conceito-chave:** são só **três operações** (mínimo, produto e combinação), e elas bastam para propagar incerteza por uma rede inteira de regras. Essa **simplicidade** foi o grande atrativo dos CFs nos anos 1970.

In [ ]:
class SistemaEspecialista:
    """
    Motor de inferência para sistema especialista com fatores de certeza.
    
    Inspirado em MYCIN (Shortliffe, 1976).
    """
    
    def __init__(self, debug: bool = False):
        self.base_conhecimento: List[Regra] = []
        self.memoria_trabalho: Dict[str, float] = {}  # fato -> CF
        self.debug = debug
        self.historico_inferencias: List[Tuple[str, str, float]] = []  # (regra, conclusao, cf)
        
    def adicionar_regra(self, regra: Regra):
        """Adiciona regra à base de conhecimento."""
        self.base_conhecimento.append(regra)
        if self.debug:
            print(f"✓ Regra adicionada: {regra.nome}")
    
    def adicionar_fato(self, nome_fato: str, cf: float = 1.0):
        """
        Adiciona ou atualiza fato na memória de trabalho.
        
        Se fato já existe, combina CFs usando fórmula de múltiplas evidências.
        """
        if nome_fato in self.memoria_trabalho:
            cf_antigo = self.memoria_trabalho[nome_fato]
            cf_novo = combinar_cf_multiplas_evidencias(cf_antigo, cf)
            self.memoria_trabalho[nome_fato] = cf_novo
            if self.debug:
                print(f"  ↻ Fato atualizado: {nome_fato} CF: {cf_antigo:.2f} → {cf_novo:.2f}")
        else:
            self.memoria_trabalho[nome_fato] = cf
            if self.debug:
                print(f"  + Fato adicionado: {nome_fato} [CF={cf:.2f}]")
    
    def obter_cf_fato(self, nome_fato: str) -> float:
        """Retorna CF de um fato (0.0 se não conhecido)."""
        return self.memoria_trabalho.get(nome_fato, 0.0)
    
    def regra_aplicavel(self, regra: Regra, limiar_cf: float = 0.2) -> Tuple[bool, float]:
        """
        Verifica se regra é aplicável.
        
        Regra aplicável se:
        - Todas premissas conhecidas
        - CF combinado das premissas > limiar
        
        Retorna: (aplicavel: bool, cf_premissas: float)
        """
        cf_premissas_list = []
        
        for premissa in regra.premissas:
            cf = self.obter_cf_fato(premissa)
            
            if cf == 0.0:  # Premissa desconhecida
                return False, 0.0
            
            cf_premissas_list.append(cf)
        
        # Combina CFs das premissas (conjunção)
        cf_combinado = combinar_cf_conjuncao(cf_premissas_list)
        
        # Regra aplicável se CF acima do limiar
        return cf_combinado >= limiar_cf, cf_combinado
    
    def disparar_regra(self, regra: Regra, cf_premissas: float):
        """
        Dispara uma regra, adicionando conclusão à memória de trabalho.
        
        CF da conclusão = CF(regra) × CF(premissas)
        """
        cf_conclusao = propagar_cf_regra(regra.cf_regra, cf_premissas)
        
        if self.debug:
            print(f"\n🔥 DISPARANDO: {regra.nome}")
            print(f"   Premissas: {' ∧ '.join(regra.premissas)} [CF={cf_premissas:.2f}]")
            print(f"   CF regra: {regra.cf_regra:.2f}")
            print(f"   Conclusão: {regra.conclusao} [CF={cf_conclusao:.2f}]")
            if regra.explicacao:
                print(f"   Explicação: {regra.explicacao}")
        
        self.adicionar_fato(regra.conclusao, cf_conclusao)
        self.historico_inferencias.append((regra.nome, regra.conclusao, cf_conclusao))
    
    def forward_chaining(self, max_iteracoes: int = 50, limiar_cf: float = 0.2):
        """
        Executa inferência forward chaining.
        
        Processo:
        1. Encontra todas regras aplicáveis
        2. Dispara regras
        3. Repete até nenhuma regra aplicável
        
        Parâmetros:
        -----------
        max_iteracoes : int
            Limite para evitar loops
        limiar_cf : float
            CF mínimo para considerar premissa verdadeira
        """
        print("\n" + "="*60)
        print("FORWARD CHAINING - ENCADEAMENTO PARA FRENTE")
        print("="*60)
        print(f"Fatos iniciais: {len(self.memoria_trabalho)}")
        print(f"Regras disponíveis: {len(self.base_conhecimento)}")
        print(f"Limiar CF: {limiar_cf}\n")
        
        regras_disparadas = set()
        iteracao = 0
        
        while iteracao < max_iteracoes:
            iteracao += 1
            
            if self.debug:
                print(f"\n{'─'*60}")
                print(f"CICLO {iteracao}")
                print(f"{'─'*60}")
            
            regras_aplicaveis = []
            
            # Encontra regras aplicáveis não disparadas
            for regra in self.base_conhecimento:
                if regra.nome in regras_disparadas:
                    continue
                
                aplicavel, cf_premissas = self.regra_aplicavel(regra, limiar_cf)
                
                if aplicavel:
                    regras_aplicaveis.append((regra, cf_premissas))
            
            if not regras_aplicaveis:
                print(f"\n{'='*60}")
                print("PARADA: Nenhuma regra aplicável")
                print(f"{'='*60}")
                break
            
            # Dispara todas regras aplicáveis
            for regra, cf_premissas in regras_aplicaveis:
                self.disparar_regra(regra, cf_premissas)
                regras_disparadas.add(regra.nome)
        
        if iteracao >= max_iteracoes:
            print(f"\n⚠ AVISO: Limite de iterações ({max_iteracoes}) atingido")
        
        self._exibir_resultados()
    
    def _exibir_resultados(self):
        """Exibe resumo dos resultados da inferência."""
        print(f"\n{'='*60}")
        print("RESULTADOS DA INFERÊNCIA")
        print(f"{'='*60}")
        
        print("\nFatos na Memória de Trabalho:")
        for fato, cf in sorted(self.memoria_trabalho.items(), key=lambda x: -x[1]):
            nivel = self._interpretar_cf(cf)
            print(f"  • {fato:30s} CF={cf:+.2f}  [{nivel}]")
        
        print(f"\nSequência de Inferências ({len(self.historico_inferencias)}):")
        for i, (regra, conclusao, cf) in enumerate(self.historico_inferencias, 1):
            print(f"  {i}. {regra:15s} → {conclusao:20s} [CF={cf:.2f}]")
        
        print(f"\n{'='*60}\n")
    
    def _interpretar_cf(self, cf: float) -> str:
        """Converte CF numérico em interpretação textual."""
        if cf > 0.8:
            return "Definitivamente"
        elif cf > 0.5:
            return "Provavelmente"
        elif cf > 0.2:
            return "Talvez"
        elif cf >= -0.2:
            return "Desconhecido"
        elif cf >= -0.5:
            return "Talvez não"
        else:
            return "Provavelmente não"
    
    def obter_diagnosticos(self, prefixo: str = "diagnostico_", min_cf: float = 0.2) -> List[Tuple[str, float]]:
        """
        Retorna diagnósticos prováveis (fatos que começam com prefixo).
        
        Retorna lista de tuplas (diagnóstico, cf) ordenada por CF decrescente.
        """
        diagnosticos = [
            (fato.replace(prefixo, ""), cf)
            for fato, cf in self.memoria_trabalho.items()
            if fato.startswith(prefixo) and cf >= min_cf
        ]
        return sorted(diagnosticos, key=lambda x: -x[1])
    
    def limpar(self):
        """Limpa memória de trabalho e histórico (mantém base de conhecimento)."""
        self.memoria_trabalho.clear()
        self.historico_inferencias.clear()


print("✓ Sistema Especialista implementado com sucesso")

#### 🔎 Como interpretar esta célula

A mensagem confirma que a classe foi criada. Os pontos-chave do motor são:

- **`regra_aplicavel`:** uma regra só dispara se **todas** as premissas são conhecidas **e** o CF combinado (o mínimo) é ≥ **limiar** (0,2). Abaixo do limiar, a evidência é considerada fraca demais. Esse limiar existia no MYCIN pelo mesmo motivo: evitar cadeias de raciocínio baseadas em quase nada.
- **`adicionar_fato`:** se a conclusão **já existe**, o CF novo é **combinado** com o antigo (`combinar_cf_multiplas_evidencias`) em vez de substituí-lo. É assim que várias regras "votam" no mesmo diagnóstico.
- **`forward_chaining`:** em cada ciclo dispara **todas** as regras aplicáveis ainda não disparadas e para quando nada novo acontece.

### 4.2 Exemplo Prático: Diagnóstico de Doenças Respiratórias

Criaremos um sistema especialista para diagnosticar:
- **Gripe (Influenza)**
- **COVID-19**
- **Resfriado Comum**
- **Pneumonia**
- **Alergia Respiratória**

Baseado em sintomas como febre, tosse, fadiga, dor de garganta, etc.

In [ ]:
# Criar sistema especialista
sistema_medico = SistemaEspecialista(debug=True)

print("="*60)
print("CONSTRUINDO BASE DE CONHECIMENTO MÉDICA")
print("="*60)
print("\nDomínio: Diagnóstico de Doenças Respiratórias\n")

# ============================================================
# REGRAS PARA GRIPE
# ============================================================

sistema_medico.adicionar_regra(Regra(
    nome="R1_Gripe",
    premissas=["febre_alta", "tosse_seca", "fadiga"],
    conclusao="diagnostico_gripe",
    cf_regra=0.85,
    explicacao="Febre alta, tosse seca e fadiga são sintomas clássicos de gripe"
))

sistema_medico.adicionar_regra(Regra(
    nome="R2_Gripe",
    premissas=["dor_muscular", "febre_alta", "inicio_subito"],
    conclusao="diagnostico_gripe",
    cf_regra=0.80,
    explicacao="Dores musculares com febre e início súbito indicam gripe"
))

sistema_medico.adicionar_regra(Regra(
    nome="R3_Gripe",
    premissas=["calafrios", "febre_alta"],
    conclusao="diagnostico_gripe",
    cf_regra=0.65,
    explicacao="Calafrios com febre alta sugerem infecção viral como gripe"
))

# ============================================================
# REGRAS PARA COVID-19
# ============================================================

sistema_medico.adicionar_regra(Regra(
    nome="R4_COVID",
    premissas=["perda_olfato", "perda_paladar", "tosse_seca"],
    conclusao="diagnostico_covid19",
    cf_regra=0.90,
    explicacao="Perda de olfato e paladar são sintomas distintivos de COVID-19"
))

sistema_medico.adicionar_regra(Regra(
    nome="R5_COVID",
    premissas=["dificuldade_respirar", "febre_alta", "fadiga"],
    conclusao="diagnostico_covid19",
    cf_regra=0.75,
    explicacao="Dificuldade respiratória com febre e fadiga podem indicar COVID-19"
))

sistema_medico.adicionar_regra(Regra(
    nome="R6_COVID",
    premissas=["contato_confirmado", "tosse_seca", "febre_alta"],
    conclusao="diagnostico_covid19",
    cf_regra=0.85,
    explicacao="Contato com caso confirmado aumenta probabilidade de COVID-19"
))

# ============================================================
# REGRAS PARA RESFRIADO COMUM
# ============================================================

sistema_medico.adicionar_regra(Regra(
    nome="R7_Resfriado",
    premissas=["coriza", "espirros", "dor_garganta_leve"],
    conclusao="diagnostico_resfriado",
    cf_regra=0.80,
    explicacao="Coriza, espirros e dor de garganta leve caracterizam resfriado"
))

sistema_medico.adicionar_regra(Regra(
    nome="R8_Resfriado",
    premissas=["tosse_leve", "coriza", "febre_baixa"],
    conclusao="diagnostico_resfriado",
    cf_regra=0.70,
    explicacao="Sintomas leves sugerem resfriado comum"
))

sistema_medico.adicionar_regra(Regra(
    nome="R9_Resfriado",
    premissas=["congestao_nasal", "espirros"],
    conclusao="diagnostico_resfriado",
    cf_regra=0.60,
    explicacao="Congestão nasal e espirros são típicos de resfriado"
))

# ============================================================
# REGRAS PARA PNEUMONIA
# ============================================================

sistema_medico.adicionar_regra(Regra(
    nome="R10_Pneumonia",
    premissas=["tosse_produtiva", "febre_alta", "dor_toracica"],
    conclusao="diagnostico_pneumonia",
    cf_regra=0.85,
    explicacao="Tosse produtiva, febre alta e dor torácica indicam pneumonia"
))

sistema_medico.adicionar_regra(Regra(
    nome="R11_Pneumonia",
    premissas=["dificuldade_respirar", "tosse_produtiva", "fadiga"],
    conclusao="diagnostico_pneumonia",
    cf_regra=0.80,
    explicacao="Dificuldade respiratória com tosse produtiva sugere pneumonia"
))

sistema_medico.adicionar_regra(Regra(
    nome="R12_Pneumonia",
    premissas=["escarro_amarelado", "febre_alta"],
    conclusao="diagnostico_pneumonia",
    cf_regra=0.75,
    explicacao="Escarro amarelado com febre indica infecção bacteriana (pneumonia)"
))

# ============================================================
# REGRAS PARA ALERGIA
# ============================================================

sistema_medico.adicionar_regra(Regra(
    nome="R13_Alergia",
    premissas=["espirros", "coceira_olhos", "coriza"],
    conclusao="diagnostico_alergia",
    cf_regra=0.85,
    explicacao="Espirros, coceira nos olhos e coriza são sintomas alérgicos"
))

sistema_medico.adicionar_regra(Regra(
    nome="R14_Alergia",
    premissas=["coceira_nariz", "espirros", "sem_febre"],
    conclusao="diagnostico_alergia",
    cf_regra=0.80,
    explicacao="Coceira nasal e espirros sem febre caracterizam alergia"
))

sistema_medico.adicionar_regra(Regra(
    nome="R15_Alergia",
    premissas=["sintomas_sazonais", "coriza"],
    conclusao="diagnostico_alergia",
    cf_regra=0.70,
    explicacao="Sintomas sazonais sugerem rinite alérgica"
))

print(f"\n✓ Base de conhecimento construída: {len(sistema_medico.base_conhecimento)} regras")

#### 🔎 Como interpretar o resultado

- A base tem **15 regras**, 3 para cada doença. Cada doença tem **várias "portas de entrada"**: gripe pode ser sugerida por febre + tosse + fadiga (R1), por dores musculares + febre + início súbito (R2) **ou** por calafrios + febre (R3).
- **Por que várias regras por doença?** Pacientes reais não apresentam todos os sintomas. Regras alternativas permitem chegar ao diagnóstico por caminhos diferentes, e quando **mais de um** caminho é satisfeito, as certezas **se somam** (veja o Caso 1).
- **Repare nos CFs das regras:** as regras com sintomas **distintivos** têm CF mais alto (R4_COVID, perda de olfato e paladar: 0,90). As com sintomas **genéricos** têm CF mais baixo (R9_Resfriado, congestão + espirros: 0,60). É assim que o especialista codifica o **poder diagnóstico** de cada combinação.

In [ ]:
print("\n" + "#"*60)
print("CASO 1: PACIENTE COM SINTOMAS DE GRIPE")
print("#"*60)

# Limpar memória de caso anterior
sistema_medico.limpar()

# Sintomas observados (com diferentes níveis de certeza)
print("\nSintomas Relatados:")
sistema_medico.adicionar_fato("febre_alta", cf=0.9)
sistema_medico.adicionar_fato("tosse_seca", cf=0.85)
sistema_medico.adicionar_fato("fadiga", cf=0.95)
sistema_medico.adicionar_fato("dor_muscular", cf=0.80)
sistema_medico.adicionar_fato("inicio_subito", cf=1.0)
sistema_medico.adicionar_fato("calafrios", cf=0.75)

# Executar inferência
sistema_medico.forward_chaining(limiar_cf=0.2)

# Obter diagnósticos
diagnosticos = sistema_medico.obter_diagnosticos()
print("\n" + "="*60)
print("DIAGNÓSTICOS PROVÁVEIS")
print("="*60)
for i, (doenca, cf) in enumerate(diagnosticos, 1):
    nivel = sistema_medico._interpretar_cf(cf)
    print(f"{i}. {doenca.upper():20s} - CF={cf:.2f} ({nivel})")


#### 🔎 Como interpretar o resultado

As **três** regras de gripe dispararam no mesmo ciclo. Acompanhe a conta:

| Regra | CF das premissas (mínimo) | × CF da regra | = contribuição | CF acumulado |
|---|:-:|:-:|:-:|:-:|
| R1_Gripe | min(0,9; 0,85; 0,95) = 0,85 | 0,85 | **0,72** | 0,72 |
| R2_Gripe | min(0,8; 0,9; 1,0) = 0,80 | 0,80 | **0,64** | 0,72 + 0,64 − 0,72×0,64 = **0,90** |
| R3_Gripe | min(0,75; 0,9) = 0,75 | 0,65 | **0,49** | 0,90 + 0,49 − 0,90×0,49 = **0,95** |

- **Resultado: GRIPE com CF 0,95, "Definitivamente".** Nenhuma regra sozinha passa de 0,72, mas três linhas de evidência **independentes** levam a certeza a 0,95.
- **Por que nenhuma outra doença aparece?** Todas as outras regras exigem pelo menos um sintoma que o paciente **não** relatou (perda de olfato, coriza, tosse produtiva...). Premissa desconhecida (CF = 0) impede o disparo.
- **Leitura da "memória de trabalho":** ela mistura **sintomas** (dados de entrada) e **conclusões** (`diagnostico_gripe`), ordenados por CF. Os sintomas mantêm os CFs informados; só as conclusões são calculadas.

In [ ]:
print("\n" + "#"*60)
print("CASO 2: PACIENTE COM SINTOMAS DE COVID-19")
print("#"*60)

sistema_medico.limpar()

print("\nSintomas Relatados:")
sistema_medico.adicionar_fato("perda_olfato", cf=1.0)
sistema_medico.adicionar_fato("perda_paladar", cf=1.0)
sistema_medico.adicionar_fato("tosse_seca", cf=0.9)
sistema_medico.adicionar_fato("febre_alta", cf=0.7)
sistema_medico.adicionar_fato("fadiga", cf=0.85)
sistema_medico.adicionar_fato("dificuldade_respirar", cf=0.6)
sistema_medico.adicionar_fato("contato_confirmado", cf=0.8)

sistema_medico.forward_chaining(limiar_cf=0.2)

diagnosticos = sistema_medico.obter_diagnosticos()
print("\n" + "="*60)
print("DIAGNÓSTICOS PROVÁVEIS")
print("="*60)
for i, (doenca, cf) in enumerate(diagnosticos, 1):
    nivel = sistema_medico._interpretar_cf(cf)
    print(f"{i}. {doenca.upper():20s} - CF={cf:.2f} ({nivel})")

#### 🔎 Como interpretar o resultado

- **COVID-19 com CF 0,96:** três regras de COVID dispararam (0,81, 0,45 e 0,59) e foram combinadas: 0,81 → 0,90 → **0,96**. A mais forte, **R4** (perda de olfato e paladar, CF da regra 0,90), sozinha já dá 0,81, porque esses sintomas são **distintivos**.
- **Gripe com CF 0,59 também aparece!** O paciente tem febre, tosse seca e fadiga, a combinação de R1_Gripe. O sistema apresenta um **diagnóstico diferencial**: COVID é muito mais provável, mas gripe não está descartada.
- **Por que gripe ficou só com 0,59?** A febre tem CF 0,7 (incerta), e pela regra do mínimo a premissa de R1 vale min(0,7; 0,9; 0,85) = 0,7, então 0,85 × 0,7 = **0,59**. Uma única medida incerta "puxa para baixo" a regra inteira.
- **💡 Lição:** um bom sistema especialista não devolve só "a resposta". Ele devolve uma **lista ordenada** de hipóteses com graus de certeza, como faz um médico. A decisão final continua sendo humana.

In [ ]:
print("\n" + "#"*60)
print("CASO 3: PACIENTE COM SINTOMAS MISTOS (INCERTEZA)")
print("#"*60)

sistema_medico.limpar()

print("\nSintomas Relatados:")
sistema_medico.adicionar_fato("febre_alta", cf=0.6)  # Febre moderada, incerto se alta
sistema_medico.adicionar_fato("tosse_seca", cf=0.7)
sistema_medico.adicionar_fato("coriza", cf=0.5)
sistema_medico.adicionar_fato("fadiga", cf=0.4)  # Fadiga leve
sistema_medico.adicionar_fato("dor_muscular", cf=0.3)

sistema_medico.forward_chaining(limiar_cf=0.2)

diagnosticos = sistema_medico.obter_diagnosticos()
print("\n" + "="*60)
print("DIAGNÓSTICOS PROVÁVEIS")
print("="*60)

if diagnosticos:
    for i, (doenca, cf) in enumerate(diagnosticos, 1):
        nivel = sistema_medico._interpretar_cf(cf)
        print(f"{i}. {doenca.upper():20s} - CF={cf:.2f} ({nivel})")
    print("\n💡 Nota: CFs baixos indicam incerteza. Exames adicionais recomendados.")
else:
    print("Nenhum diagnóstico conclusivo. Sintomas insuficientes ou incertos.")

#### 🔎 Como interpretar o resultado

- **Só uma regra disparou (R1_Gripe), com CF 0,34, "Talvez".** O CF da premissa é min(0,6; 0,7; **0,4**) = 0,4, limitado pela fadiga fraca, e 0,85 × 0,4 = 0,34.
- **Por que R2_Gripe não disparou?** Ela exige `inicio_subito`, que não foi informado. Mesmo com dor muscular (0,3) e febre presentes, falta uma premissa.
- **Por que nenhum resfriado, se há coriza (0,5)?** As regras de resfriado exigem **também** espirros, dor de garganta leve ou febre baixa. A coriza **sozinha** não basta para nenhuma regra.
- **Leitura clínica:** com sintomas vagos, o sistema **não força** um diagnóstico confiante. Ele sinaliza baixa certeza e recomenda exames. É o comportamento desejável: **reconhecer a própria incerteza**.
- **⚠️ Limitação revelada:** um paciente com 5 sintomas recebeu apenas uma hipótese fraca porque as regras exigem **combinações exatas** de sintomas. Um especialista humano consideraria o quadro como um todo. Essa **rigidez** é uma das limitações clássicas dos sistemas baseados em regras.

### 4.3 Observações sobre a Implementação

**Pontos-chave do sistema implementado:**

1. **Separação de Conhecimento e Inferência:**
   - Base de conhecimento (regras) é independente do motor de inferência
   - Novas regras podem ser adicionadas sem modificar código
   - Permite colaboração com especialistas do domínio

2. **Propagação de Incerteza:**
   - Fórmulas de CF do MYCIN implementadas corretamente
   - Combinação de premissas via MIN (conjunção)
   - Propagação via multiplicação (CF_regra × CF_premissas)
   - Múltiplas evidências combinadas adequadamente

3. **Rastreabilidade:**
   - Histórico de inferências mantido
   - Possível explicar POR QUE cada conclusão foi atingida
   - Debug mode mostra raciocínio passo a passo

4. **Tratamento de Casos Realistas:**
   - **Caso 1:** Sintomas claros → diagnóstico com alta certeza
   - **Caso 2:** Sintomas distintivos (perda olfato/paladar) → COVID-19
   - **Caso 3:** Sintomas ambíguos → múltiplos diagnósticos com baixa certeza

5. **Limitações Aparentes:**
   - Não considera relações temporais ("febre por 3 dias")
   - Não modela dependências entre sintomas
   - CF não são probabilidades verdadeiras (não somam a 1)
   - Aquisição de conhecimento ainda requer programação

**Comparação com MYCIN Original:**

| **Aspecto** | **Nossa Implementação** | **MYCIN (1976)** |
|-------------|------------------------|------------------|
| **Regras** | ~15 regras | ~600 regras |
| **Domínio** | Doenças respiratórias | Infecções sanguíneas |
| **Inferência** | Forward chaining | Backward chaining |
| **Explicação** | Traço básico | Sistema WHY/HOW sofisticado |
| **Interface** | Programática | Diálogo em inglês |
| **Terapia** | Não implementada | Recomendava antibióticos e doses |

**Melhorias Possíveis:**
- Adicionar backward chaining (raciocínio dirigido por metas)
- Implementar módulo de explicação interativo
- Suportar perguntas ao usuário (sintomas desconhecidos)
- Interface gráfica para construção de regras
- Integração com bancos de dados médicos

### 4.4 🎬 Animação: a certeza sendo construída regra a regra (caso COVID-19)

No Caso 2, quatro regras dispararam. A animação mostra **cada disparo**, o cálculo do CF da conclusão e a **combinação** com as evidências anteriores.

In [ ]:
import contextlib, io

def rodar_caso(sintomas, sistema=None):
    # Executa o sistema médico em silêncio e devolve (diagnósticos, histórico de inferências).
    s = sistema or sistema_medico
    debug_original, s.debug = s.debug, False
    s.limpar()
    for nome, cf in sintomas.items():
        if cf != 0:
            s.adicionar_fato(nome, cf)
    with contextlib.redirect_stdout(io.StringIO()):
        s.forward_chaining(limiar_cf=0.2)
    s.debug = debug_original
    return s.obter_diagnosticos(min_cf=-1), list(s.historico_inferencias)


CASO_COVID = {'perda_olfato': 1.0, 'perda_paladar': 1.0, 'tosse_seca': 0.9, 'febre_alta': 0.7,
              'fadiga': 0.85, 'dificuldade_respirar': 0.6, 'contato_confirmado': 0.8}
_, hist_covid = rodar_caso(CASO_COVID)

# 🎬 ANIMAÇÃO — a certeza de cada diagnóstico sendo construída regra a regra
regras_por_nome = {r.nome: r for r in sistema_medico.base_conhecimento}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={'width_ratios': [1, 1.3]})
doencas = ['gripe', 'covid19', 'resfriado', 'pneumonia', 'alergia']

def quadro_cf(k):
    ax1.clear(); ax2.clear(); ax2.axis('off')
    cfs = {d: 0.0 for d in doencas}
    for nome, concl, cf in hist_covid[:k]:
        d = concl.replace('diagnostico_', '')
        cfs[d] = cf if cfs[d] == 0 else comb(cfs[d], cf)
    ax1.barh(doencas, [cfs[d] for d in doencas], color=CORES['verde'])
    for lim, txt in [(0.2, 'talvez'), (0.5, 'provável'), (0.8, 'definitivo')]:
        ax1.axvline(lim, color='gray', ls=':'); ax1.text(lim, 4.6, txt, fontsize=8, ha='center')
    ax1.set_xlim(0, 1); ax1.set_ylim(-0.6, 5); ax1.set_xlabel('CF do diagnóstico')
    ax1.set_title(f'Após {k} regra(s) disparada(s)')
    if k == 0:
        ax2.text(0, 0.5, 'Paciente: perda de olfato e paladar, tosse seca,\nfebre (0.7), fadiga, falta de ar (0.6), contato confirmado',
                 fontsize=11, va='center'); return
    nome, concl, cf = hist_covid[k - 1]
    r = regras_por_nome[nome]
    cf_prem = cf / r.cf_regra
    linhas = [f'{nome}: SE {" ∧ ".join(r.premissas)}', f'        ENTÃO {concl}',
              '', f'CF(premissas) = min(...) = {cf_prem:.2f}',
              f'CF(conclusão) = CF(regra) × CF(premissas) = {r.cf_regra:.2f} × {cf_prem:.2f} = {cf:.2f}']
    d = concl.replace('diagnostico_', '')
    anteriores = [c for n, cc, c in hist_covid[:k - 1] if cc == concl]
    if anteriores:
        antes = anteriores[0]
        for c in anteriores[1:]:
            antes = comb(antes, c)
        linhas.append(f'Combinando com o CF anterior: {antes:.2f} + {cf:.2f} − {antes:.2f}×{cf:.2f} = {comb(antes, cf):.2f}')
    ax2.text(0, 0.95, '\n'.join(linhas), fontsize=10, va='top', family='monospace')

anim = animation.FuncAnimation(fig, quadro_cf, frames=len(hist_covid) + 1, interval=1500)
mostrar_animacao(anim)

#### 🔎 Como interpretar a animação

- **Esquerda:** a barra de cada doença cresce à medida que as regras disparam. As linhas pontilhadas marcam os limiares de interpretação (0,2 "talvez", 0,5 "provável", 0,8 "definitivo").
- **Direita:** a conta exata do disparo atual: (1) o **mínimo** das premissas, (2) o **produto** pelo CF da regra e (3), se já havia evidência para a mesma doença, a **combinação** com o valor anterior.
- **Sequência:**
  1. **R1_Gripe** → gripe 0,59;
  2. **R4_COVID** → COVID 0,81 (de cara já passa de "definitivo");
  3. **R5_COVID** → contribuição 0,45, e a combinação leva COVID a **0,90**;
  4. **R6_COVID** → contribuição 0,59, e COVID chega a **0,96**.
- **O que observar:** os **saltos ficam menores** a cada evidência (0,81 → 0,90 → 0,96), o mesmo efeito de retornos decrescentes do gráfico da Seção 3.6. A ordem dos disparos não muda o resultado final, porque a combinação é comutativa e associativa (comprovado na Seção 4.7).

### 4.5 A base de conhecimento como uma rede

Uma forma útil de **revisar** uma base de regras com especialistas é desenhá-la como uma rede.

In [ ]:
# A base de conhecimento como uma REDE: sintomas (esquerda) → diagnósticos (direita)
regras = sistema_medico.base_conhecimento
sintomas = sorted({p for r in regras for p in r.premissas})
diag = sorted({r.conclusao for r in regras})
ys = {s: i for i, s in enumerate(sintomas)}
yd = {d: (i + 0.5) * len(sintomas) / len(diag) - 0.5 for i, d in enumerate(diag)}
cor_d = dict(zip(diag, [CORES['azul'], CORES['vermelho'], CORES['verde'], CORES['laranja'], CORES['roxo']]))

fig, ax = plt.subplots(figsize=(12, 8))
for r in regras:
    for p in r.premissas:
        ax.plot([0, 1], [ys[p], yd[r.conclusao]], color=cor_d[r.conclusao], alpha=0.5, lw=1 + 3 * (r.cf_regra - 0.6))
uso = {s: sum(s in r.premissas for r in regras) for s in sintomas}
for s in sintomas:
    ax.text(-0.01, ys[s], f'{s} ({uso[s]})', ha='right', va='center', fontsize=9,
            fontweight='bold' if uso[s] >= 4 else 'normal')
for d in diag:
    ax.text(1.01, yd[d], d.replace('diagnostico_', '').upper(), ha='left', va='center', fontsize=11,
            color=cor_d[d], fontweight='bold')
ax.set_xlim(-0.45, 1.3); ax.axis('off')
ax.set_title('Base de conhecimento: cada linha liga uma premissa à conclusão da regra (espessura ∝ CF da regra)\n'
             'entre parênteses: em quantas regras o sintoma aparece')
plt.show()

#### 🔎 Como interpretar a figura

- Cada **linha** liga um sintoma (premissa) à doença concluída pela regra. A **cor** identifica a doença e a **espessura** é proporcional ao CF da regra.
- **`febre_alta (7)`** é o sintoma mais conectado: aparece em 7 regras, de gripe, COVID **e** pneumonia. Sozinho, ele **não discrimina** entre doenças: é um sintoma **sensível, mas pouco específico**.
- **Sintomas com uma única linha** (`perda_olfato`, `escarro_amarelado`, `coceira_olhos`...) são os mais **discriminativos**: quando presentes, apontam para uma doença só.
- **Por que desenhar a base?** Com especialistas, essa visualização ajuda a encontrar:
  - doenças com poucas "portas de entrada" (cobertura fraca);
  - sintomas importantes que **não** aparecem em nenhuma regra;
  - regras que só diferem em um detalhe (possível redundância, veja a Seção 6.4).

### 4.6 🎛️ Consultório virtual: análise de sensibilidade

Mova os *sliders* (0 = sintoma ausente, 1 = sintoma certamente presente) e observe como os diagnósticos reagem. É uma **análise de sensibilidade**: descobrir quais evidências mais influenciam cada conclusão.

In [ ]:
# 🎛️ Consultório virtual: ajuste a certeza de cada sintoma e veja os diagnósticos mudarem
SINTOMAS_UI = ['febre_alta', 'tosse_seca', 'fadiga', 'dor_muscular', 'perda_olfato', 'perda_paladar',
               'coriza', 'espirros', 'coceira_olhos', 'dificuldade_respirar', 'tosse_produtiva', 'dor_toracica']

def consultorio(**sintomas):
    diags, hist = rodar_caso(sintomas)
    cfs = {d: cf for d, cf in diags}
    fig, ax = plt.subplots(figsize=(8, 3))
    vals = [cfs.get(d, 0) for d in doencas]
    ax.barh(doencas, vals, color=[CORES['verde'] if v >= 0.5 else CORES['laranja'] if v >= 0.2 else CORES['cinza'] for v in vals])
    for lim in (0.2, 0.5, 0.8):
        ax.axvline(lim, color='gray', ls=':')
    ax.set_xlim(0, 1); ax.set_title('Diagnósticos (CF)')
    plt.show()
    print('Regras disparadas:', [f'{n} ({cf:.2f})' for n, _, cf in hist] or 'nenhuma')

sliders = {s: widgets.FloatSlider(min=0, max=1, step=0.1, value=0.0, description=s[:14],
                                  continuous_update=False, style={'description_width': '110px'})
           for s in SINTOMAS_UI}
sliders['febre_alta'].value, sliders['tosse_seca'].value, sliders['fadiga'].value = 0.9, 0.8, 0.7
interact(consultorio, **sliders);

#### 🔎 Como usar e interpretar

O consultório começa com febre (0,9), tosse seca (0,8) e fadiga (0,7), e o sistema aponta **gripe 0,59**. Experimente:

1. **Suba `perda_olfato` e `perda_paladar` para 1,0:** COVID dispara (R4). Repare que a gripe **continua** lá: as regras não "competem" entre si, cada doença é avaliada de forma independente.
2. **Abaixe `fadiga` para 0,1:** a gripe **desaparece**. O mínimo das premissas cai abaixo do limiar 0,2 e R1 deixa de disparar. Um único sintoma fraco pode **bloquear** uma regra inteira (efeito da regra do mínimo).
3. **Zere tudo e suba `espirros`, `coceira_olhos` e `coriza`:** alergia aparece. Com `coriza` mais `espirros` sozinhos, nada aparece (falta dor de garganta para R7). Que regra precisaria existir?
4. **Volte ao estado inicial e mova só `febre_alta`:** entre 0,2 e 0,7 a gripe cresce junto com a febre; **acima de 0,7 ela para de subir**, porque o mínimo passa a ser a fadiga (0,7); e **abaixo de 0,2 ela desaparece de repente** (limiar). Esse comportamento "em platô" é típico da regra do mínimo.

- **💡 Conceito-chave, análise de sensibilidade:** mostra **quais evidências realmente importam** para cada conclusão. Na engenharia do conhecimento, é usada para validar a base com especialistas: "faz sentido que a gripe dependa tanto da fadiga?".

### 4.7 Limitações dos fatores de certeza na prática

Os CFs foram criados para serem **simples e intuitivos**, e por isso fazem suposições fortes. Em 1986, **David Heckerman** mostrou que a combinação do MYCIN só é coerente com a teoria da probabilidade se as evidências forem **condicionalmente independentes**. Vejamos o que acontece quando isso não vale.

In [ ]:
print('=' * 70)
print('PROBLEMA 1 — Regra duplicada: a MESMA evidência contada duas vezes')
print('=' * 70)
base = rodar_caso({'febre_alta': 0.9, 'tosse_seca': 0.85, 'fadiga': 0.95})[0]
print(f'Base original     → gripe CF = {dict(base)["gripe"]:.2f}')

duplicada = SistemaEspecialista()
for r in sistema_medico.base_conhecimento:
    duplicada.adicionar_regra(r)
r1 = sistema_medico.base_conhecimento[0]
duplicada.adicionar_regra(Regra('R1_Gripe_copia', list(r1.premissas), r1.conclusao, r1.cf_regra))
dup = rodar_caso({'febre_alta': 0.9, 'tosse_seca': 0.85, 'fadiga': 0.95}, sistema=duplicada)[0]
print(f'Com R1 duplicada  → gripe CF = {dict(dup)["gripe"]:.2f}   ← nenhuma informação nova, mas a certeza subiu!')

print('\n' + '=' * 70)
print('PROBLEMA 2 — Evidências dependentes (febre e calafrios andam juntos)')
print('=' * 70)
p_h = 0.10                                   # prevalência da gripe
p_e_h, p_e_nh = 0.90, 0.20                   # P(febre | gripe), P(febre | não gripe)
post_1 = p_e_h * p_h / (p_e_h * p_h + p_e_nh * (1 - p_h))
post_ingenuo = (p_e_h ** 2 * p_h) / (p_e_h ** 2 * p_h + p_e_nh ** 2 * (1 - p_h))
print(f'P(gripe) a priori                             = {p_h:.3f}')
print(f'P(gripe | febre)                              = {post_1:.3f}')
print(f'P(gripe | febre, calafrios) se calafrios só acompanha a febre (redundante) = {post_1:.3f}')
print(f'P(gripe | febre, calafrios) tratando como INDEPENDENTES                     = {post_ingenuo:.3f}  ← exagero')
print(f'CF: duas regras com CF 0.5 cada → {comb(0.5, 0.5):.2f}  (a fórmula do MYCIN também assume independência)')

print('\n' + '=' * 70)
print('PROPRIEDADE DESEJÁVEL — a ordem das evidências não importa (comutativa e associativa)')
print('=' * 70)
from itertools import permutations
evid = [0.6, -0.3, 0.45]
for ordem in permutations(evid):
    acc = ordem[0]
    for c in ordem[1:]:
        acc = comb(acc, c)
    print(f'ordem {ordem} → CF = {acc:.4f}')

#### 🔎 Como interpretar o resultado

**Problema 1, regra duplicada:** copiar a regra R1 elevou a certeza de gripe de **0,72 para 0,92**, sem **nenhuma** informação nova sobre o paciente. O motor trata as duas cópias como evidências **independentes** e as combina. Em bases grandes, redundâncias assim surgem sem querer (dois especialistas escrevendo regras parecidas) e inflam certezas silenciosamente. A Seção 6.4 mostra como detectá-las.

**Problema 2, evidências dependentes:**
- Com probabilidades, saber que há febre leva P(gripe) de 0,10 para **0,33**.
- Se "calafrios" só aparece **junto** com febre (é redundante), ele **não traz informação nova**: a probabilidade correta continua **0,33**.
- Tratando as duas como independentes, o resultado vai para **0,69**, mais que o **dobro** do correto.
- A combinação de CFs faz a mesma suposição de independência: 0,5 e 0,5 viram 0,75.
- **Por que isso acontece?** Heckerman (1986) mostrou que a fórmula do MYCIN equivale a uma atualização probabilística **somente** se as evidências forem condicionalmente independentes dada a hipótese. Na medicina isso raramente é verdade: febre, calafrios e dor no corpo **andam juntos**.

**Propriedade desejável:** as 6 ordens possíveis das evidências (0,6; −0,3; 0,45) dão **exatamente** 0,6857. A combinação é **comutativa e associativa**, então o resultado não depende da ordem em que as regras disparam. Essa foi uma exigência explícita de Shortliffe e Buchanan.

- **💡 Conclusão:** os CFs são **práticos** e funcionaram bem no MYCIN, porque as decisões de terapia eram pouco sensíveis a pequenas variações de CF. Mas não têm a **fundamentação** da probabilidade. Por isso foram substituídos por **redes bayesianas** (Pearl, 1988), que representam explicitamente as dependências entre variáveis (Aula 16).

## 5. Estudo de Caso: MYCIN

### 5.1 Contexto e Desenvolvimento

**MYCIN** foi desenvolvido entre 1972-1980 por **Edward Shortliffe** como parte de sua tese de doutorado em Stanford, sob orientação de **Bruce Buchanan** e **Edward Feigenbaum**.

**Problema abordado:**
- Diagnóstico de infecções sanguíneas (bacteremia) e meningite
- Recomendação de terapia antibiótica
- Cálculo de dosagem baseado em peso do paciente e função renal

**Por que esse domínio?**
1. Infecções sanguíneas requerem tratamento imediato (horas são críticas)
2. Resultados de cultura demoram 24-48h
3. Médicos devem iniciar terapia empírica baseada em evidências indiretas
4. Escolha errada de antibiótico pode ser fatal

### 5.2 Arquitetura do MYCIN

**Componentes principais:**

1. **Base de Conhecimento:** ~600 regras de produção
2. **Motor de Inferência:** Backward chaining (raciocínio dirigido por metas)
3. **Módulo de Explicação:** Sistema WHY/HOW para justificar raciocínio
4. **Subsistema de Aquisição:** TEIRESIAS (permitia especialistas adicionarem regras)

**Exemplo de regra MYCIN (simplificada):**

```
RULE052
IF:    1) The site of the culture is blood, AND
       2) The gram stain of the organism is gramneg, AND
       3) The morphology of the organism is rod, AND
       4) The patient has been seriously burned
THEN:  There is suggestive evidence (0.4) that the identity 
       of the organism is pseudomonas
```

### 5.3 Fatores de Certeza no MYCIN

**Motivação para CF (em vez de probabilidades):**

1. **Dificuldade de elicitar probabilidades:**
   - Especialistas não pensam naturalmente em termos de P(A|B)
   - Probabilidades condicionais são anti-intuitivas
   - Difícil obter P(A), P(B), P(A∧B) consistentes

2. **Custo computacional:**
   - Teorema de Bayes requer P(E|H) E P(H|E)
   - Atualização bayesiana completa é cara (1970s)
   - Independência condicional raramente válida

3. **Interpretabilidade:**
   - CF de 0.7 mais intuitivo que "70% de probabilidade"
   - Escala [-1, 1] natural para especialistas

**Fórmulas de MYCIN detalhadas:**

#### Medida de Crença e Descrença

CF decomposto em:
```
MB(H, E) = Measure of Belief (Medida de Crença)
MD(H, E) = Measure of Disbelief (Medida de Descrença)

CF(H, E) = MB(H, E) - MD(H, E)
```

**Definições originais:**

```
MB(H, E) = 
  ⎧ 1                           se P(H) = 1
  ⎨ max[P(H|E) - P(H), 0]      
  ⎩ ――――――――――――――――――――――       caso contrário
        1 - P(H)

MD(H, E) = 
  ⎧ 1                           se P(H) = 0
  ⎨ max[P(H) - P(H|E), 0]      
  ⎩ ――――――――――――――――――――――       caso contrário
          P(H)
```

**Propriedades:**
- MB e MD ∈ [0, 1]
- MB e MD nunca são ambos não-zero
- CF ∈ [-1, 1]
- CF = 0 significa P(H|E) = P(H) (evidência não altera crença)

### 5.4 Exemplo de Consulta MYCIN

**Diálogo típico (traduzido):**

```
MYCIN: Nome do paciente?
USUÁRIO: John Doe

MYCIN: Sexo?
USUÁRIO: Masculino

MYCIN: Idade?
USUÁRIO: 35 anos

MYCIN: Há informação sobre cultura?
USUÁRIO: Sim

MYCIN: Local da cultura?
USUÁRIO: Sangue

MYCIN: POR QUÊ
[Usuário pergunta por que a questão foi feita]

MYCIN: Estou tentando determinar organismos significativos
       que requerem terapia. Preciso saber local da cultura
       para aplicar regras sobre tipos de organismos.
       [RULE033: SE local é sangue E gram é negativo ENTÃO...]

USUÁRIO: Sangue

MYCIN: Gram da cultura?
USUÁRIO: Negativo

[... mais perguntas ...]

MYCIN: Minhas recomendações terapêuticas:

       TERAPIA 1:
       Droga: Gentamicina
       Dose: 119 mg (1.7 mg/kg) q8h IV
       Comentários: Modificar dose se função renal alterada

       TERAPIA 2:
       Droga: Clindamicina  
       Dose: 595 mg (8.5 mg/kg) q6h IV

USUÁRIO: COMO 1
[Usuário pergunta como conclusão foi alcançada]

MYCIN: Gentamicina foi selecionada porque:
       1. RULE095 concluiu organismo é E.coli (CF=0.8)
       2. RULE003 estabeleceu gentamicina cobre E.coli (CF=0.95)
       3. Paciente não tem contraindicações
       4. CF combinado: 0.76 ("Provável")
```

### 5.5 Resultados e Avaliação

**Performance clínica (Yu et al., 1979):** em uma avaliação **cega**, 8 especialistas em doenças infecciosas de fora de Stanford julgaram as prescrições para **10 casos de meningite**, sem saber quem as tinha feito (MYCIN ou médicos).

| **Prescritor** | **Prescrições consideradas aceitáveis** |
|---|---|
| MYCIN | **65%** |
| Professores de Stanford especialistas em infectologia | 42,5% a 62,5% |
| Médico residente e estudante de Medicina | também avaliados, com aceitação menor que a do MYCIN |

> ℹ️ Os percentuais acima são os mais citados na literatura sobre o estudo. Consulte o artigo original (JAMA, 242(12), 1979) para a tabela completa.

**Achados surpreendentes:**
1. MYCIN superou médicos não-especialistas
2. Terapia de MYCIN considerada tão boa quanto especialistas
3. Sistema sempre explicava seu raciocínio (médicos nem sempre)

**Por que nunca foi usado clinicamente?**

1. **Questões legais e éticas:**
   - Quem é responsável se sistema erra?
   - Médicos relutantes em confiar em computador
   - Regulamentação de software médico inexistente

2. **Problemas práticos:**
   - Entrada de dados demorada (30-45 min por consulta)
   - Computadores caros e raros em hospitais (1970s)
   - Interface textual complexa

3. **Limitações técnicas:**
   - Não aprendia com experiência
   - Base de conhecimento difícil de manter
   - Não integrado com sistemas hospitalares

### 5.6 Legado do MYCIN

Apesar de nunca usado clinicamente, MYCIN teve impacto enorme:

1. **Conceitual:**
   - Provou que IA pode atingir performance expert
   - Demonstrou importância da explicação
   - Estabeleceu engenharia de conhecimento como disciplina

2. **Técnico:**
   - Fatores de certeza amplamente adotados
   - EMYCIN (shell derivado) usado em centenas de sistemas
   - Arquitetura influenciou gerações de sistemas especialistas

3. **Sistemas derivados:**
   - **EMYCIN:** Shell genérico (Empty MYCIN)
   - **GUIDON:** Sistema tutor baseado em MYCIN
   - **NEOMYCIN:** Redesign com representação explícita de estratégias
   - **TEIRESIAS:** Sistema de aquisição de conhecimento

**Em resumo:** o MYCIN mostrou que um programa podia atingir desempenho de especialista em um domínio restrito, mas também revelou os desafios de colocar IA em uso clínico real. Seu legado mais duradouro são as ideias sobre **representação do conhecimento**, **raciocínio sob incerteza** e a importância da **explicação**.

## 6. Sistemas Especialistas Modernos

### 6.1 Declínio e Renascimento

**Anos 1990-2000: "Inverno dos Sistemas Especialistas"**

Limitações que causaram declínio:
1. **Aquisição de conhecimento:** Bottleneck — extrair conhecimento de especialistas é caro e lento
2. **Manutenção:** Bases grandes (1000+ regras) são frágeis e difíceis de atualizar
3. **Brittleness:** Performance degrada fora do domínio restrito
4. **Falta de aprendizado:** Não melhoram com experiência
5. **Escalabilidade:** Dificuldade em combinar conhecimento de múltiplos especialistas

**Anos 2010-2020: Renascimento Parcial**

Fatores que renovaram interesse:
1. **Explicabilidade (XAI):** Machine learning "caixa-preta" criou demanda por IA transparente
2. **Regulamentação:** GDPR e legislações exigem explicação de decisões automatizadas
3. **Sistemas híbridos:** Combinar ML (reconhecimento) com regras (raciocínio)
4. **Motores modernos:** Drools, CLIPS, Jess continuam relevantes

### 6.2 Drools (Motor de Regras de Negócio)

**Drools** é um motor de regras de negócio (Business Rules Management System - BRMS) open-source desenvolvido em Java.

**Características:**
- Implementa algoritmo RETE otimizado
- Suporta forward e backward chaining
- Integração com Java/Spring
- DSL (Domain-Specific Language) para regras
- Amplamente usado em bancos e seguradoras

**Exemplo de regra Drools:**

```drools
rule "Aprovação de Empréstimo - Risco Baixo"
when
    $cliente : Cliente( 
        score >= 750, 
        rendaMensal >= 5000,
        historicoInadimplencia == false 
    )
    $emprestimo : Emprestimo( 
        valor <= 50000,
        cliente == $cliente 
    )
then
    modify($emprestimo) {
        setAprovado(true),
        setTaxaJuros(1.2),
        setJustificativa("Cliente baixo risco - taxa preferencial")
    }
    System.out.println("Empréstimo aprovado: " + $emprestimo);
end
```

**Aplicações Drools:**
- Aprovação de crédito e empréstimos
- Detecção de fraude
- Precificação dinâmica
- Roteamento de processos
- Validação de transações

### 6.3 PyKnow (Sistemas Especialistas em Python)

**PyKnow** é uma biblioteca Python para criar sistemas especialistas, inspirada em CLIPS.

**Exemplo PyKnow:**

In [ ]:
# Nota: PyKnow não está instalado por padrão e não é mais mantido.
# O fork "experta" (pip install experta) mantém a mesma API: troque "from pyknow import *" por "from experta import *".
# 
# Este é um exemplo da estrutura PyKnow (não executável sem instalação)

exemplo_pyknow = '''
from pyknow import *

class SistemaAnimal(KnowledgeEngine):
    """Sistema especialista para classificação de animais."""

    @Rule(Fact(tem_pelos=True),
          Fact(amamenta=True))
    def mamifero(self):
        self.declare(Fact(classe="mamífero"))
        print("→ É um mamífero")

    @Rule(Fact(classe="mamífero"),
          Fact(come_carne=True),
          Fact(dentes_afiados=True))
    def carnivoro(self):
        self.declare(Fact(tipo="carnívoro"))
        print("→ É um carnívoro")

    @Rule(Fact(tipo="carnívoro"),
          Fact(cor="laranja"),
          Fact(tem_listras=True))
    def tigre(self):
        self.declare(Fact(animal="tigre"))
        print("→ É um TIGRE!")

# Uso:
sistema = SistemaAnimal()
sistema.reset()

# Adicionar fatos
sistema.declare(Fact(tem_pelos=True))
sistema.declare(Fact(amamenta=True))
sistema.declare(Fact(come_carne=True))
sistema.declare(Fact(dentes_afiados=True))
sistema.declare(Fact(cor="laranja"))
sistema.declare(Fact(tem_listras=True))

# Executar inferência
sistema.run()
'''

print("="*60)
print("ESTRUTURA DE SISTEMA ESPECIALISTA COM PyKnow")
print("="*60)
print(exemplo_pyknow)
print("\n✓ PyKnow permite sintaxe declarativa elegante para regras")
print("✓ Decoradores @Rule facilitam leitura")
print("✓ Integração natural com Python")

#### 🔎 Como interpretar esta célula

- A célula apenas **imprime** um exemplo de código (não o executa), porque a biblioteca não vem instalada.
- **Compare com o nosso sistema:** em vez de listas de premissas, o PyKnow/experta usa **decoradores** `@Rule(...)` sobre métodos Python. A **ação** da regra é o corpo do método, que pode fazer qualquer coisa (declarar fatos, imprimir, chamar APIs).
- O motor dessas bibliotecas segue o **CLIPS**: *forward chaining* com algoritmo **RETE** (Aula 07), resolução de conflito por prioridade (`salience`) e casamento de padrões com **variáveis** (`Fact(cor=MATCH.c)`), algo que nossas implementações didáticas não suportam.

### 6.4 Engenharia do conhecimento: verificando a base de regras automaticamente

Bases com centenas de regras acumulam defeitos difíceis de ver a olho. Ferramentas de **verificação e validação (V&V)** procuram padrões suspeitos:

| Defeito | Exemplo | Por que é ruim |
|---|---|---|
| **Redundância** | duas regras com as mesmas premissas e conclusão | conta a mesma evidência duas vezes |
| **Conflito** | mesmas premissas, CFs com sinais opostos | o sistema "discorda de si mesmo" |
| **Subsunção** | `SE A E B ENTÃO X` e `SE A E B E C ENTÃO X` | a mais específica sempre dispara junto com a geral (dupla contagem) |
| **Ciclo** | `X → Y` e `Y → X` | raciocínio circular; pode inflar certezas ou gerar *loops* |

In [ ]:
from collections import defaultdict

def verificar_base(regras, nome_base):
    print('=' * 70, f'\nRELATÓRIO DE VERIFICAÇÃO — {nome_base} ({len(regras)} regras)\n' + '=' * 70)
    problemas = 0
    # 1) Regras redundantes (mesmas premissas e mesma conclusão)
    vistas = {}
    for r in regras:
        chave = (frozenset(r.premissas), r.conclusao, r.cf_regra > 0)
        if chave in vistas:
            print(f'⚠️  REDUNDÂNCIA: {r.nome} repete {vistas[chave]} (risco de dupla contagem de evidência)')
            problemas += 1
        vistas.setdefault(chave, r.nome)
    # 2) Conflitos (mesmas premissas, conclusões diferentes com sinais opostos de CF)
    por_premissas = defaultdict(list)
    for r in regras:
        por_premissas[frozenset(r.premissas)].append(r)
    for prem, rs in por_premissas.items():
        for a in rs:
            for b in rs:
                if a.nome < b.nome and a.conclusao == b.conclusao and a.cf_regra * b.cf_regra < 0:
                    print(f'❌ CONFLITO: {a.nome} e {b.nome} têm as mesmas premissas e CFs de sinais opostos')
                    problemas += 1
    # 3) Regras subsumidas (premissas de uma contêm as de outra com a mesma conclusão)
    for a in regras:
        for b in regras:
            if a is not b and a.conclusao == b.conclusao and set(a.premissas) < set(b.premissas):
                print(f'ℹ️  SUBSUNÇÃO: {b.nome} ⊃ {a.nome} — sempre que {b.nome} dispara, {a.nome} também dispara')
                problemas += 1
    # 4) Ciclos (A → B → ... → A)
    grafo = defaultdict(set)
    for r in regras:
        for p in r.premissas:
            grafo[p].add(r.conclusao)
    def tem_ciclo(no, caminho):
        for viz in grafo[no]:
            if viz in caminho:
                return caminho[caminho.index(viz):] + [viz]
            c = tem_ciclo(viz, caminho + [viz])
            if c:
                return c
    ciclos = {}
    for n in list(grafo):
        c = tem_ciclo(n, [n])
        if c:
            ciclos.setdefault(frozenset(c), c)       # o mesmo ciclo começando em outro nó conta uma vez
    ciclos = ciclos.values()
    for c in ciclos:
        print(f'🔁 CICLO: {" → ".join(c)}')
        problemas += 1
    # 5) Sintomas que só aparecem em uma regra (possível lacuna de conhecimento)
    contagem = defaultdict(int)
    for r in regras:
        for p in r.premissas:
            contagem[p] += 1
    raros = sorted(p for p, n in contagem.items() if n == 1)
    print(f'ℹ️  Premissas usadas em uma única regra ({len(raros)}): {", ".join(raros)}')
    print(f'\nTotal de alertas: {problemas}\n')


verificar_base(sistema_medico.base_conhecimento, 'Base médica do notebook')

base_com_defeitos = [
    Regra('D1', ['febre_alta', 'tosse_seca'], 'diagnostico_gripe', 0.7),
    Regra('D2', ['tosse_seca', 'febre_alta'], 'diagnostico_gripe', 0.7),
    Regra('D3', ['febre_alta', 'tosse_seca', 'fadiga'], 'diagnostico_gripe', 0.8),
    Regra('D4', ['espirros'], 'diagnostico_alergia', 0.6),
    Regra('D5', ['espirros'], 'diagnostico_alergia', -0.4),
    Regra('D6', ['diagnostico_gripe'], 'imunidade_baixa', 0.5),
    Regra('D7', ['imunidade_baixa'], 'diagnostico_gripe', 0.6),
]
verificar_base(base_com_defeitos, 'Base com defeitos plantados')

#### 🔎 Como interpretar o resultado

**Base médica do notebook:** **0 alertas**: não há regras repetidas, conflitantes, subsumidas nem ciclos. O relatório lista **16 premissas usadas em uma única regra**. Isso não é erro, mas merece revisão: se o paciente tem `calafrios` sem `febre_alta`, essa informação é **ignorada**. O especialista deveria confirmar que é isso mesmo que deseja.

**Base com defeitos plantados:** o verificador encontrou todos os defeitos inseridos:

| Alerta | Defeito | Consequência se não corrigido |
|---|---|---|
| D2 repete D1 | Mesmas premissas em outra ordem | Dupla contagem: gripe com certeza inflada (como no Problema 1) |
| D4 × D5 | `espirros` → alergia com CF +0,6 **e** −0,4 | O sistema "discorda de si mesmo"; o resultado depende de detalhes da fórmula |
| D3 ⊃ D1, D2 | D3 é D1 + `fadiga` | Sempre que D3 dispara, D1 também, e a fadiga é contada "por cima" de evidência já contada |
| Ciclo gripe ↔ imunidade | D6 e D7 se alimentam | Raciocínio circular: "tem gripe porque tem imunidade baixa porque tem gripe" |

- **💡 Conceito-chave:** em engenharia de software testamos **código**; em engenharia do conhecimento também precisamos testar o **conhecimento**. Ferramentas de V&V como esta foram criadas nos anos 1980 justamente porque bases como a do XCON (milhares de regras) tinham se tornado impossíveis de revisar à mão.

## 7. Comparação: Sistemas Especialistas vs ML vs Sistemas de Produção

### 7.1 Tabela Comparativa Detalhada

| **Critério** | **Sistemas de Produção** | **Sistemas Especialistas** | **Machine Learning** |
|--------------|--------------------------|---------------------------|---------------------|
| **Conhecimento** | Regras procedurais gerais | Conhecimento heurístico de domínio | Padrões aprendidos de dados |
| **Aquisição** | Programação direta | Engenharia de conhecimento | Aprendizado automático |
| **Representação** | Regras SE-ENTÃO | Regras + CF + Frames | Pesos, árvores, vetores |
| **Incerteza** | Geralmente determinístico | CF, fuzzy, probabilidades | Probabilidades estatísticas |
| **Explicação** | Traço de execução | WHY/HOW detalhado | Difícil ("caixa-preta") |
| **Aprendizado** | Não | Não | Sim (essencial) |
| **Manutenção** | Fácil (código) | Difícil (muitas regras) | Fácil (re-treinar) |
| **Performance** | Depende das regras | Nível expert (em domínio) | Pode superar humanos |
| **Escalabilidade** | Boa | Limitada (~1000 regras) | Excelente |
| **Generalização** | Baixa | Muito baixa (brittleness) | Alta |
| **Dados requeridos** | Nenhum | Conhecimento explícito | Grandes volumes |
| **Velocidade** | Rápida | Rápida | Depende (treinamento lento) |
| **Garantias** | Formais possíveis | Parciais | Estatísticas apenas |
| **Custo desenvolvimento** | Baixo | Alto (especialistas) | Médio-Alto (dados) |
| **Exemplo clássico** | GPS, SOAR | MYCIN, XCON | AlphaGo, GPT |

### 7.2 Quando Usar Cada Abordagem?

**Use Sistemas de Produção quando:**
- ✓ Problema pode ser decomposto em regras simples
- ✓ Raciocínio procedural suficiente
- ✓ Não há incerteza significativa
- ✓ Velocidade de execução crítica
- **Exemplos:** Controle de semáforos, jogos simples, automação residencial

**Use Sistemas Especialistas quando:**
- ✓ Domínio bem definido e restrito
- ✓ Conhecimento explícito disponível (livros, especialistas)
- ✓ Explicação é obrigatória (regulamentação, confiança)
- ✓ Poucos dados históricos disponíveis
- ✓ Conhecimento muda lentamente
- **Exemplos:** Diagnóstico médico raro, configuração de sistemas, análise regulatória

**Use Machine Learning quando:**
- ✓ Grandes volumes de dados disponíveis
- ✓ Padrões complexos difíceis de explicitar
- ✓ Conhecimento implícito (difícil verbalizar)
- ✓ Ambiente muda rapidamente
- ✓ Performance > Explicabilidade
- **Exemplos:** Reconhecimento de imagem, processamento de linguagem, detecção de fraude

**Use Abordagem Híbrida quando:**
- ✓ Combinar percepção (ML) com raciocínio (regras)
- ✓ Precisão de ML + Explicabilidade de regras
- ✓ Restrições regulatórias sobre parte do sistema
- **Exemplos:** Veículos autônomos (visão ML + regras de trânsito), diagnóstico médico assistido, sistemas financeiros

### 7.3 Vantagens e Desvantagens

#### Sistemas Especialistas

**Vantagens:**
1. ✅ **Transparência:** Raciocínio compreensível e auditável
2. ✅ **Conhecimento explícito:** Regras podem ser lidas, validadas, certificadas
3. ✅ **Poucos dados:** Não requer grandes datasets
4. ✅ **Consistência:** Sempre toma mesma decisão com mesmos inputs
5. ✅ **Preservação de expertise:** Captura conhecimento de especialistas
6. ✅ **Explicação:** Sistema pode justificar decisões
7. ✅ **Garantias:** Verificação formal possível em casos simples

**Desvantagens:**
1. ❌ **Aquisição de conhecimento:** Bottleneck — caro e demorado
2. ❌ **Brittleness:** Performance colapsa fora do domínio
3. ❌ **Não aprende:** Não melhora com experiência
4. ❌ **Escalabilidade:** Difícil gerenciar >1000 regras
5. ❌ **Manutenção:** Regras podem conflitar; difícil debugar
6. ❌ **Subjetividade:** Diferentes especialistas = diferentes regras
7. ❌ **Domínio limitado:** Funciona apenas em área específica

#### Machine Learning

**Vantagens:**
1. ✅ **Aprendizado automático:** Extrai padrões de dados sem programação explícita
2. ✅ **Generalização:** Funciona em casos não vistos
3. ✅ **Escalabilidade:** Melhora com mais dados
4. ✅ **Adaptação:** Pode ser re-treinado para novos padrões
5. ✅ **Performance:** Frequentemente supera humanos

**Desvantagens:**
1. ❌ **Caixa-preta:** Difícil explicar decisões (especialmente deep learning)
2. ❌ **Dados:** Requer grandes volumes de dados rotulados
3. ❌ **Viés:** Herda preconceitos presentes nos dados
4. ❌ **Adversarial:** Vulnerável a ataques adversariais
5. ❌ **Custo:** Treinamento pode ser caro (GPU, tempo)

### 7.4 Convergência: Sistemas Neurossimbólicos

**Tendência atual:** Combinar melhor dos dois mundos

**Arquiteturas híbridas:**

```
┌──────────────────────────────────────┐
│      SISTEMA NEUROSSIMBÓLICO         │
├──────────────────────────────────────┤
│                                      │
│   ┌────────────┐    ┌────────────┐  │
│   │   NEURAL   │───►│  SIMBÓLICO │  │
│   │  (Percepção)│    │(Raciocínio)│  │
│   │            │    │            │  │
│   │ • CNN      │    │ • Regras   │  │
│   │ • RNN      │    │ • Lógica   │  │
│   │ • Attention│    │ • Ontologias│  │
│   └────────────┘    └────────────┘  │
│         ▲                 │          │
│         └─────────────────┘          │
│           Feedback Loop              │
└──────────────────────────────────────┘
```

**Exemplos:**
- **IBM Watson for Oncology:** ML para análise de textos médicos + regras para recomendações
- **Veículos autônomos:** Deep learning para visão + regras para decisões de trânsito
- **AlphaGo:** Redes neurais para avaliação + busca em árvore (Monte Carlo)

**Referência importante:**

Na palestra *From System 1 Deep Learning to System 2 Deep Learning* (NeurIPS 2019), **Yoshua Bengio** defendeu que a próxima geração de IA precisa combinar o reconhecimento rápido de padrões ("Sistema 1", em que as redes neurais são fortes) com o raciocínio deliberado, composicional e explicável ("Sistema 2", tradicionalmente o terreno dos sistemas simbólicos). Essa é uma das motivações centrais para os sistemas neurossimbólicos.

## 8. Aplicações de Sistemas Especialistas

### 8.1 Medicina e Saúde

**Sistemas Clássicos:**

1. **MYCIN (1976):** Diagnóstico de infecções sanguíneas
   - Performance: 65% precisão (equivalente a especialistas)
   - Legado: Estabeleceu padrões para sistemas médicos

2. **CADUCEUS/INTERNIST-I (1982):** Diagnóstico médico geral
   - Base: ~500 doenças, ~3500 sintomas
   - Abordagem: Raciocínio abdutor e eliminação

3. **PUFF (1979):** Interpretação de testes de função pulmonar
   - Desenvolvido em Stanford usando EMYCIN
   - Usado clinicamente por anos

4. **DXplain (1984-presente):** Sistema de diagnóstico da Harvard Medical School
   - Ainda em uso! Atualizado continuamente
   - ~2400 doenças, ~5000 achados clínicos
   - Disponível online para médicos

**Sistemas Modernos:**

- **Isabel Healthcare:** Sistema de diagnóstico diferencial usado em hospitais
- **Watson for Oncology (IBM):** Recomendações de tratamento de câncer
- **UpToDate:** Base de conhecimento médico com sistema de busca inteligente

### 8.2 Finanças e Seguros

**Análise de Crédito:**
```
Regras típicas:
- SE score_credito > 750 E renda/divida < 0.3 ENTÃO aprovar_limite_alto
- SE inadimplencia_historica E valor > R$50k ENTÃO rejeitar
- SE primeiro_emprestimo E garantia_real ENTÃO CF_aprovacao = 0.8
```

**Detecção de Fraude:**
- Regras baseadas em padrões conhecidos
- Transações fora do perfil usual
- Combinação com ML para novos padrões

**Compliance e Regulamentação:**
- Verificação automática de conformidade
- Análise de transações suspeitas (AML - Anti-Money Laundering)
- Sistemas explicáveis requeridos por lei

**Sistemas Reais:**
- **XCON (1980):** Configuração de computadores DEC - economizou $40M/ano
- **Drools:** Usado por bancos para regras de negócio
- **FICO Scoring:** Baseado parcialmente em regras especialistas

### 8.3 Indústria e Manufatura

**Diagnóstico de Falhas:**
```
Exemplo - Sistema de Manutenção de Turbinas:
- SE vibracao_alta E temperatura_normal ENTÃO desbalanceamento_rotor
- SE temperatura_alta E vibracao_normal ENTÃO problema_lubrificacao
- SE ruido_metalico E vibracao_irregular ENTÃO falha_rolamento
```

**Controle de Processos:**
- Refinarias de petróleo
- Fábricas químicas
- Usinas de energia

**Configuração e Design:**
- **XCON/R1:** Configuração de VAX (DEC) - ~2500 regras
- Sistemas CAD inteligentes
- Otimização de layout de fábrica

**Sistemas Notáveis:**
- **DELTA (1986):** Manutenção de locomotivas diesel (GE)
- **CATS-1 (1987):** Diagnóstico de máquinas em fábricas

### 8.4 Jurídico e Regulatório

**Análise de Contratos:**
- Identificação de cláusulas problemáticas
- Verificação de conformidade
- Revisão de due diligence

**Sistemas Jurídicos:**
- **TAXMAN (1977):** Planejamento tributário corporativo
- **HYPO (1987):** Raciocínio baseado em casos legais
- **ROSS (IBM Watson):** Pesquisa jurídica assistida por IA

**Compliance Automatizado:**
```
Regras GDPR (exemplo simplificado):
- SE processar_dados_pessoais E NOT consentimento_explicito 
  ENTÃO violacao_GDPR
- SE reter_dados > 90_dias E NOT justificativa_legal 
  ENTÃO exigir_exclusao
```

### 8.5 Agricultura e Meio Ambiente

**Diagnóstico de Pragas e Doenças:**
```
Exemplo:
- SE manchas_amarelas_folhas E clima_umido E cultura=soja 
  ENTÃO ferrugem_asiatica [CF=0.7]
- SE praga_identificada E estagio=vegetativo 
  ENTÃO recomendar(inseticida_tipo_A, dose=2L/ha)
```

**Sistemas Notáveis:**
- **PLANT/ds (1984):** Diagnóstico de doenças de plantas (Illinois)
- **POMME (1984):** Gestão de pomares
- **COMAX (1989):** Gestão de cultivo de algodão

**Gestão Ambiental:**
- Previsão de impacto ambiental
- Gestão de recursos hídricos
- Planejamento de reflorestamento

### 8.6 Outras Aplicações Relevantes

**Educação:**
- **GUIDON (1979):** Sistema tutor baseado em MYCIN
- **SCHOLAR (1970):** Tutor de geografia
- Sistemas adaptativos de aprendizado

**Telecomunicações:**
- **ACE (1986):** Análise de falhas em cabos telefônicos (AT&T)
- Diagnóstico de redes
- Otimização de roteamento

**Exploração Mineral:**
- **PROSPECTOR (1979):** Avaliação de depósitos minerais
  - Famoso por prever depósito de molibdênio de $100M
- Análise geológica

**Aeroespacial:**
- Diagnóstico de satélites
- Planejamento de missões
- Controle de sistemas críticos

### 8.7 Requisitos para Sucesso

Sistemas especialistas funcionam melhor quando:

✅ **Domínio adequado:**
- Escopo bem definido e limitado
- Conhecimento explícito disponível
- Problema requer expertise humana

✅ **Especialistas disponíveis:**
- Experts dispostos a colaborar
- Conhecimento pode ser articulado
- Consenso sobre decisões corretas

✅ **Valor demonstrável:**
- Economiza tempo/dinheiro
- Melhora consistência
- Preserva conhecimento crítico

✅ **Aspectos práticos:**
- Interface amigável
- Integração com sistemas existentes
- Manutenção viável
- Explicação satisfaz usuários

## 9. Exercícios Propostos

### Exercício 1 — Sistema de Recomendação de Investimentos (Nível Básico)

Implemente um sistema especialista para recomendar tipo de investimento baseado no perfil do investidor.

**Especificação:**

- **Entrada:** Idade, renda mensal, objetivo (aposentadoria, curto prazo, reserva), tolerância a risco
- **Saída:** Recomendações de investimento (CDB, Tesouro, Ações, Fundos) com CF

**Regras sugeridas:**
```
SE idade < 30 E tolerancia_risco = alta ENTÃO recomendar_acoes [CF=0.8]
SE objetivo = reserva_emergencia ENTÃO recomendar_cdb_liquidez [CF=0.9]
SE idade > 55 E objetivo = aposentadoria ENTÃO recomendar_renda_fixa [CF=0.85]
SE renda > 10000 E tolerancia_risco = media ENTÃO recomendar_fundos_multimercado [CF=0.7]
```

**Tarefas:**
1. Crie pelo menos 12 regras cobrindo diferentes perfis
2. Implemente sistema usando classe `SistemaEspecialista`
3. Teste com 3 perfis diferentes (conservador, moderado, agressivo)
4. Sistema deve combinar múltiplas evidências usando CFs
5. Exiba explicação de por que cada investimento foi recomendado

---

### Exercício 2 — Diagnóstico Automotivo com Múltiplas Evidências (Nível Intermediário)

Crie sistema especialista para diagnóstico de problemas automotivos que **combina múltiplas evidências** para mesma conclusão.

**Cenário:**
Mecânico observa sintomas e sistema calcula CF para cada problema possível, combinando evidências de múltiplas regras.

**Sintomas possíveis:**
```python
motor_nao_liga, dificuldade_dar_partida, barulho_ao_ligar,
luz_bateria_acesa, farois_fracos, motor_falhando,
fumaca_escape, superaquecimento, vazamento_oleo,
barulho_metalico, vibracao_excessiva
```

**Problemas possíveis:**
```python
bateria_fraca, alternador_defeituoso, motor_fundido,
velas_gastas, bomba_combustivel, correia_alternador
```

**Tarefas:**
1. Crie pelo menos 15 regras onde múltiplas regras podem concluir mesmo problema
2. Implemente sistema que **combina CFs** de múltiplas evidências
3. Teste caso onde 3+ regras apontam para mesmo problema
4. Mostre como CF aumenta à medida que mais evidências aparecem
5. Compare diagnóstico com sintomas parciais vs completos

**Exemplo esperado:**
```
REGRA 1: SE motor_nao_liga E luz_bateria_acesa ENTÃO bateria_fraca [CF=0.6]
REGRA 2: SE farois_fracos E dificuldade_partida ENTÃO bateria_fraca [CF=0.5]
REGRA 3: SE bateria_fraca E motor_liga_as_vezes ENTÃO alternador_defeituoso [CF=0.7]

Resultado: bateria_fraca com CF combinado de ~0.8 (duas evidências)
```

---

### Exercício 3 — Módulo de Explicação WHY/HOW (Nível Intermediário)

Estenda a classe `SistemaEspecialista` para incluir **módulo de explicação** que responde:
- **"POR QUÊ uma conclusão foi alcançada?"**
- **"COMO uma decisão foi tomada?"**

**Especificação:**

1. Adicione método `explicar_conclusao(fato)` que:
   - Lista todas regras que contribuíram para o fato
   - Mostra CF de cada regra e CF combinado final
   - Exibe premissas que satisfizeram cada regra

2. Adicione método `justificar_pergunta(sintoma)` que:
   - Mostra quais regras usam aquele sintoma
   - Explica que conclusões podem ser alcançadas

**Exemplo de saída esperada:**
```
>>> sistema.explicar_conclusao('diagnostico_gripe')

EXPLICAÇÃO: Como 'diagnostico_gripe' foi concluído?

Três regras contribuíram:

1. REGRA R1_Gripe [CF=0.85]
   Premissas satisfeitas:
     • febre_alta [CF=0.9]
     • tosse_seca [CF=0.85]
     • fadiga [CF=0.95]
   CF premissas combinado: 0.85 (mínimo)
   CF conclusão: 0.85 × 0.85 = 0.72

2. REGRA R2_Gripe [CF=0.80]
   Premissas satisfeitas:
     • dor_muscular [CF=0.80]
     • febre_alta [CF=0.9]
     • inicio_subito [CF=1.0]
   CF premissas combinado: 0.80
   CF conclusão: 0.80 × 0.80 = 0.64

CF FINAL (combinando evidências):
  CF(R1, R2) = 0.72 + 0.64 - (0.72 × 0.64) = 0.90

Conclusão: DEFINITIVAMENTE gripe [CF=0.90]
```

**Tarefas:**
1. Implemente métodos de explicação
2. Teste com sistema médico do notebook
3. Gere explicações para pelo menos 3 diagnósticos

---

### Exercício 4 — Backward Chaining (Nível Avançado)

Implemente **backward chaining** (encadeamento para trás) no sistema especialista.

**Diferença de Forward Chaining:**
- **Forward:** Dos fatos às conclusões (dados → metas)
- **Backward:** Da meta aos fatos (meta → dados)

**Especificação:**

1. Adicione método `backward_chaining(meta, limiar_cf=0.2)`
2. Sistema deve:
   - Tentar provar a meta
   - Buscar regras que concluem a meta
   - Recursivamente tentar provar premissas dessas regras
   - Perguntar ao usuário sobre fatos desconhecidos

**Pseudocódigo:**
```python
def provar(meta):
    if meta in memoria_trabalho:
        return CF(meta)
    
    para cada regra onde conclusao == meta:
        cf_premissas = []
        para cada premissa:
            cf_p = provar(premissa)  # Recursão!
            se cf_p < limiar:
                falha
            cf_premissas.append(cf_p)
        
        cf_conclusao = propagar_cf(regra.cf, min(cf_premissas))
        adicionar_fato(meta, cf_conclusao)
        return cf_conclusao
    
    # Se chegou aqui, perguntar ao usuário
    return perguntar_usuario(meta)
```

**Tarefas:**
1. Implemente backward chaining completo
2. Teste com sistema médico tentando provar "diagnostico_gripe"
3. Compare com forward chaining:
   - Número de regras avaliadas
   - Ordem de raciocínio
   - Eficiência (quando há muitas regras)
4. Implemente sistema interativo que pergunta sintomas conforme necessário

---

### Exercício 5 — Sistema Híbrido com Machine Learning (Nível Avançado)

Crie um **sistema híbrido** que combina machine learning com regras especialistas.

**Arquitetura:**
```
Dados brutos → ML (classificação inicial) → Regras (refinamento + explicação) → Decisão final
```

**Cenário: Aprovação de Empréstimo**

**Parte 1 — ML:**
- Treinar modelo (árvore de decisão ou regressão logística)
- Entrada: idade, renda, score_credito, dividas, historico
- Saída: probabilidade de aprovação

**Parte 2 — Regras Especialistas:**
```
SE prob_ml > 0.8 E NOT inadimplencia_recente ENTÃO aprovar [CF=0.9]
SE prob_ml > 0.6 E valor < 10000 E garantia ENTÃO aprovar [CF=0.7]
SE fraude_suspeita ENTÃO rejeitar [CF=1.0]  # Override ML!
SE cliente_vip E prob_ml > 0.4 ENTÃO aprovar [CF=0.8]
```

**Tarefas:**
1. Gere dataset sintético de empréstimos (500+ exemplos)
2. Treine modelo ML (sklearn)
3. Implemente regras especialistas que:
   - Usam probabilidade ML como entrada
   - Aplicam regras de negócio
   - Podem overrule ML em casos críticos
4. Compare:
   - Acurácia: ML puro vs Híbrido
   - Explicabilidade: ML vs Híbrido
   - Casos onde regras corrigem ML
5. Implemente explicação: "Aprovado porque ML=0.85 E cliente sem histórico negativo (Regra R5)"

**Vantagens esperadas:**
- Performance de ML + Transparência de regras
- Regras capturam políticas que ML não aprende
- Sistema auditável e regulável

## 10. Referências Bibliográficas

### Livros Clássicos

1. **Russell, S. & Norvig, P.** (2020). *Artificial Intelligence: A Modern Approach* (4ª ed.). Pearson. — Capítulos 12 e 13 sobre raciocínio sob incerteza e sistemas especialistas.

2. **Buchanan, B. G. & Shortliffe, E. H.** (1984). *Rule-Based Expert Systems: The MYCIN Experiments of the Stanford Heuristic Programming Project*. Addison-Wesley. — Documentação definitiva do MYCIN.

3. **Jackson, P.** (1999). *Introduction to Expert Systems* (3ª ed.). Addison-Wesley. — Texto introdutório abrangente sobre sistemas especialistas.

4. **Giarratano, J. & Riley, G.** (2005). *Expert Systems: Principles and Programming* (4ª ed.). Thomson. — Guia prático com foco em CLIPS.

5. **Durkin, J.** (1994). *Expert Systems: Design and Development*. Prentice Hall. — Metodologia de desenvolvimento de sistemas especialistas.

6. **Hayes-Roth, F., Waterman, D. A., & Lenat, D. B.** (Eds.). (1983). *Building Expert Systems*. Addison-Wesley. — Coletânea clássica sobre construção de sistemas especialistas.

7. **Nilsson, N. J.** (1998). *Artificial Intelligence: A New Synthesis*. Morgan Kaufmann. — Capítulos sobre representação de conhecimento e sistemas baseados em regras.

### Artigos Fundamentais sobre MYCIN e Fatores de Certeza

8. **Shortliffe, E. H.** (1976). *Computer-Based Medical Consultations: MYCIN*. Elsevier. — Tese de doutorado publicada como livro.

9. **Shortliffe, E. H. & Buchanan, B. G.** (1975). *A Model of Inexact Reasoning in Medicine*. Mathematical Biosciences, 23(3-4), 351-379. — Introdução aos fatores de certeza.

10. **Yu, V. L., et al.** (1979). *Antimicrobial Selection by a Computer: A Blinded Evaluation by Infectious Disease Experts*. JAMA, 242(12), 1279-1282. — Avaliação clínica do MYCIN.

11. **Heckerman, D.** (1986). *Probabilistic Interpretations for MYCIN's Certainty Factors*. In *Uncertainty in Artificial Intelligence* (pp. 167-196). Elsevier. — Análise formal dos CFs.

### Artigos sobre Outros Sistemas Clássicos

12. **Feigenbaum, E. A., Buchanan, B. G., & Lederberg, J.** (1971). *On Generality and Problem Solving: A Case Study Using the DENDRAL Program*. Machine Intelligence, 6, 165-190. — DENDRAL, primeiro sistema especialista.

13. **McDermott, J.** (1982). *R1: A Rule-Based Configurer of Computer Systems*. Artificial Intelligence, 19(1), 39-88. — XCON/R1, primeiro sucesso comercial.

14. **Duda, R. O., Gaschnig, J., & Hart, P. E.** (1979). *Model Design in the PROSPECTOR Consultant System for Mineral Exploration*. In *Expert Systems in the Micro-Electronic Age* (pp. 153-167). Edinburgh University Press. — PROSPECTOR.

15. **Pople, H. E.** (1982). *Heuristic Methods for Imposing Structure on Ill-Structured Problems*. In *Artificial Intelligence in Medicine* (pp. 119-190). Westview Press. — CADUCEUS/INTERNIST.

### Engenharia de Conhecimento e Aquisição

16. **Feigenbaum, E. A.** (1977). *The Art of Artificial Intelligence: Themes and Case Studies of Knowledge Engineering*. IJCAI-77. — Cunhou termo "engenharia de conhecimento".

17. **Davis, R.** (1979). *Interactive Transfer of Expertise: Acquisition of New Inference Rules*. Artificial Intelligence, 12(2), 121-157. — TEIRESIAS, sistema de aquisição.

18. **Clancey, W. J.** (1987). *Knowledge-Based Tutoring: The GUIDON Program*. MIT Press. — Sistema tutor baseado em MYCIN.

### Raciocínio sob Incerteza

19. **Pearl, J.** (1988). *Probabilistic Reasoning in Intelligent Systems: Networks of Plausible Inference*. Morgan Kaufmann. — Redes bayesianas como alternativa a CFs.

20. **Zadeh, L. A.** (1965). *Fuzzy Sets*. Information and Control, 8(3), 338-353. — Artigo original de lógica fuzzy.

21. **Shafer, G.** (1976). *A Mathematical Theory of Evidence*. Princeton University Press. — Teoria Dempster-Shafer.

### Sistemas Modernos e Aplicações

22. **Forgy, C. L.** (1982). *Rete: A Fast Algorithm for the Many Pattern/Many Object Pattern Match Problem*. Artificial Intelligence, 19(1), 17-37. — Algoritmo RETE usado em sistemas modernos.

23. **Proctor, M. et al.** (2013). *Drools Documentation* (versão 6.0). Red Hat/JBoss. — Motor de regras moderno.

24. **Miller, R. A., Pople, H. E., & Myers, J. D.** (1982). *Internist-I, an Experimental Computer-Based Diagnostic Consultant for General Internal Medicine*. New England Journal of Medicine, 307(8), 468-476.

### Críticas e Limitações

25. **Dreyfus, H. L. & Dreyfus, S. E.** (1986). *Mind over Machine: The Power of Human Intuition and Expertise in the Era of the Computer*. Free Press. — Crítica filosófica aos sistemas especialistas.

26. **Lenat, D. B. & Guha, R. V.** (1990). *Building Large Knowledge-Based Systems: Representation and Inference in the Cyc Project*. Addison-Wesley. — Aborda problema de conhecimento de senso comum.

### Sistemas Neurossimbólicos (Integração com ML)

27. **Marcus, G.** (2020). *The Next Decade in AI: Four Steps Towards Robust Artificial Intelligence*. arXiv:2002.06177. — Argumento por sistemas híbridos.

28. **Bengio, Y., Lecun, Y., & Hinton, G.** (2021). *Deep Learning for AI*. Communications of the ACM, 64(7), 58-65. — Perspectiva sobre futuro da IA.

29. **Kautz, H.** (2022). *The Third AI Summer: AAAI Robert S. Engelmore Memorial Lecture*. AI Magazine, 43(1), 105-125. — Ressurgimento de abordagens simbólicas.

### Recursos Online

30. **CLIPS Official Website:** http://www.clipsrules.net/ — C Language Integrated Production System.

31. **Drools Documentation:** https://www.drools.org/ — Motor de regras open-source.

32. **PyKnow GitHub:** https://github.com/buguroo/pyknow — Biblioteca Python para sistemas especialistas.

---

**Notebook preparado para a disciplina de Inteligência Artificial.**  
**Curso de Ciência da Computação.**  
**Tópico: Sistemas Especialistas Baseados em Regras.**